In [1]:
# 0

import warnings
from pathlib import Path

import pandas as pd
import numpy as np

from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.metrics import roc_auc_score

warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)


In [2]:
# 1

XLSX = 'kospi_data.xlsx'

EXCLUDE_FIN = ['A000680', 'A000880', 'A005110', 'A008060', 'A015020', 'A023590',
               'A026890', 'A030190', 'A034310', 'A210980', 'A244920']

def build_gd(path=XLSX, out='gd.parquet'):
    if Path(out).exists():
        df = pd.read_parquet(out)
        print(f'{out} loaded. {len(df)}행  {df["qtr"].min()} ~ {df["qtr"].max()}')
        return df
    raw = pd.read_excel(path, sheet_name='34_gd', header=None)
    body = raw.iloc[14:, :2].dropna()
    df = pd.DataFrame({
        'qtr': pd.PeriodIndex(body.iloc[:, 0].astype(str), freq='Q'),
        'gd': pd.to_numeric(body.iloc[:, 1], errors='coerce'),
    }).dropna().sort_values('qtr').reset_index(drop=True)
    df.to_parquet(out)
    print(f'{out} written. {len(df)}행  {df["qtr"].min()} ~ {df["qtr"].max()}')
    return df

financials = pd.read_parquet('financials.parquet')
financials = financials[~financials['code'].isin(EXCLUDE_FIN)].reset_index(drop=True)

prices = pd.read_parquet('prices.parquet')
prices = prices[~prices['code'].isin(EXCLUDE_FIN)].reset_index(drop=True)

delisted = pd.read_parquet('delisted.parquet')
delisted = delisted[~delisted['code'].isin(EXCLUDE_FIN)].reset_index(drop=True)

gd = build_gd()

print(f'financials  {financials.shape}  종목 {financials["code"].nunique()}')
print(f'prices      {prices.shape}  종목 {prices["code"].nunique()}')
print(f'delisted    {len(delisted)}개')
print(f'columns: {financials.columns.tolist()}')

gd.parquet loaded. 105행  2000Q1 ~ 2026Q1
financials  (116235, 29)  종목 1107
prices      (353133, 5)  종목 1107
delisted    355개
columns: ['code', 'date', 'actq', 'rectq', 'invtq', 'ppentq', 'atq', 'lctq', 'dlttq', 'dlcq', 'ltq', 'seqq', 'cogsq', 'xsgaq', 'saleq', 'apq', 'opinc', 'xintq', 'ibq', 'txditcq', 'pstkq', 'cheq', 'oancfq', 'dpq', 'dvq', 'cstkq', 'req', 'epsq', 'txtq']


In [3]:
# 2

FORCE_IMPAIR = ['A001600', 'A006210']
FORCE_REORG = ['A019930']
BAD_DEFAULT = ['A011840', 'A018590', 'A001980', 'A010730', 'A005600',
               'A007050', 'A011800', 'A002130']
BAD_SUSPEND = ['A000200', 'A003810', 'A017120', 'A000310', 'A004510',
               'A004550', 'A004740', 'A012600', 'A093230']
FORMAL = ['A003980', 'A012720', 'A009790', 'A012270', 'A008750', 'A003020',
          'A009760', 'A006250', 'A001190', 'A016160', 'A015110', 'A002530',
          'A017320', 'A014080', 'A008500']
MERGER_EXTRA = ['A103160', 'A049770', 'A008000', 'A002250', 'A027390', 'A144620',
                'A015350', 'A138250', 'A068400', 'A001880', 'A003410', 'A115390',
                'A034300', 'A010420', 'A450140', 'A019440']
NAN_CODES = ['A950010', 'A011160', 'A102280']

BAD_CATS = ['감사의견', '자본잠식', '정리절차', '부도', '실질심사']
NONBAD_CATS = ['합병', '자진', '형식요건']

d = delisted.copy()
d['cat'] = '미분류'
d.loc[d['reason_cat'] == '감사의견', 'cat'] = '감사의견'
d.loc[d['reason_cat'] == '자본잠식', 'cat'] = '자본잠식'
d.loc[d['reason_cat'] == '정리절차', 'cat'] = '정리절차'
d.loc[d['reason_cat'] == '합병·해산', 'cat'] = '합병'
d.loc[d['reason_cat'] == '자진·신청', 'cat'] = '자진'
d.loc[d['code'].isin(FORCE_IMPAIR), 'cat'] = '자본잠식'
d.loc[d['code'].isin(FORCE_REORG), 'cat'] = '정리절차'
d.loc[d['code'].isin(BAD_DEFAULT), 'cat'] = '부도'
d.loc[d['code'].isin(BAD_SUSPEND), 'cat'] = '실질심사'
d.loc[d['code'].isin(FORMAL), 'cat'] = '형식요건'
d.loc[d['code'].isin(MERGER_EXTRA), 'cat'] = '합병'
d.loc[d['code'].isin(NAN_CODES), 'cat'] = '미분류'

d['is_bad'] = d['cat'].isin(BAD_CATS).astype(int)
d['excl'] = (d['cat'] == '미분류').astype(int)
d.to_parquet('delisted_cat.parquet')

fin_has = financials.groupby('code')['atq'].apply(lambda s: s.notna().any())
fin_codes = set(fin_has[fin_has].index)
h = d[d['code'].isin(fin_codes)]

cnt = pd.DataFrame({
    '전체': d['cat'].value_counts(),
    '재무존재': h['cat'].value_counts(),
}).reindex(BAD_CATS + NONBAD_CATS + ['미분류']).fillna(0).astype(int)
cnt['구분'] = np.where(cnt.index.isin(BAD_CATS), '부실',
                     np.where(cnt.index.isin(NONBAD_CATS), '비부실', '미분류'))

blocks = [cnt[cnt['구분'] == g].sort_values('재무존재', ascending=False)
          for g in ['부실', '비부실', '미분류']]
tbl = pd.concat(blocks).rename_axis('사유').reset_index()[['구분', '사유', '전체', '재무존재']]
display(tbl)

sub = tbl.groupby('구분', sort=False)[['전체', '재무존재']].sum().reindex(['부실', '비부실', '미분류'])
sub.loc['합계'] = sub.sum()
display(sub)

hb = h[h['is_bad'] == 1].copy()
hb['year'] = hb['del_date'].dt.year.astype(int)
yr = (hb.groupby(['year', 'cat']).size().unstack('cat', fill_value=0)
      .reindex(columns=blocks[0].index.tolist(), fill_value=0))
yr['합계'] = yr.sum(axis=1)
yr.loc['합계'] = yr.sum()
display(yr)

YR0, YR1 = 2002, 2025

excl_codes = set(d.query('excl == 1')['code'])
dcx = d[d['excl'] == 0][['code', 'del_date', 'is_bad']].copy()
dcx['del_year'] = dcx['del_date'].dt.year.astype(int)

pxr = prices.copy()
pxr['year'] = pd.to_datetime(pxr['date']).dt.year
live = pxr[pxr['ret'].notna() | pxr['mktcap'].notna()]
alive = live[['code', 'year']].drop_duplicates()
alive = alive[(alive['year'] >= YR0) & (alive['year'] <= YR1)]
alive = alive[~alive['code'].isin(excl_codes)]
alive = alive.merge(dcx[['code', 'del_year', 'is_bad']], on='code', how='left')
alive = alive[alive['del_year'].isna() | (alive['year'] <= alive['del_year'])]
alive['y'] = ((alive['del_year'] == alive['year']) & (alive['is_bad'] == 1)).astype(int)
alive['censored'] = ((alive['del_year'] == alive['year']) & (alive['is_bad'] == 0)).astype(int)

riskset = (alive.rename(columns={'year': 'obs_year'})[['code', 'obs_year', 'y', 'censored']]
           .sort_values(['code', 'obs_year']).reset_index(drop=True))
riskset.to_parquet('riskset.parquet')

rs = pd.DataFrame({'값': [
    len(riskset), riskset['code'].nunique(),
    f"{int(riskset['obs_year'].min())}~{int(riskset['obs_year'].max())}",
    int(riskset['y'].sum()), int(riskset['censored'].sum())]},
    index=pd.Index(['위험집합 행', '종목', '기간', '사건', '절단'], name='항목'))
display(rs)

,구분,사유,전체,재무존재
0,부실,감사의견,71,70
1,부실,자본잠식,31,23
2,부실,정리절차,31,10
3,부실,실질심사,9,9
4,부실,부도,8,8
5,비부실,합병,130,83
6,비부실,형식요건,15,15
7,비부실,자진,23,14
8,미분류,미분류,37,3


,전체,재무존재
구분,,
부실,150,120
비부실,168,112
미분류,37,3
합계,355,235


cat,감사의견,자본잠식,정리절차,실질심사,부도,합계
year,,,,,,
2000,0,0,1,0,0,1
2001,0,0,4,4,1,9
2002,15,4,1,0,0,20
2003,2,1,1,0,1,5
2004,5,1,1,1,3,11
2005,5,2,2,0,1,10
2006,1,0,0,0,0,1
2007,1,4,0,0,0,5
2008,1,0,0,0,0,1


,값
항목,
위험집합 행,16657
종목,962
기간,2002~2025
사건,102
절단,80


In [4]:
# 3

HALT_N = 8
FULL_HALT = 0.95
EXTREME = 100.0
SENS_N = [5, 8, 10, 15]

d = pd.read_parquet('daily_returns.parquet')
d['date'] = pd.to_datetime(d['date'])
d = d.sort_values(['code', 'date']).reset_index(drop=True)
d['ym'] = pd.PeriodIndex(d['date'], freq='M')
d['year'] = d['date'].dt.year

d['is0'] = (d['dret'] == 0).astype(int)
d['grp'] = (d['is0'] != d.groupby('code')['is0'].shift()).cumsum()
d['runlen_yr'] = d.groupby(['grp', 'year'])['is0'].transform('size')
d['halt'] = ((d['is0'] == 1) & (d['runlen_yr'] >= HALT_N)).astype(int)
d['extreme'] = (d['dret'].abs() > EXTREME).astype(int)

d[['code', 'date', 'ym', 'year', 'dret', 'halt', 'extreme']].to_parquet('halt_daily.parquet')

hm = (d.groupby(['code', 'ym'])
      .agg(n_days=('halt', 'size'), n_halt=('halt', 'sum'),
           n_ext=('extreme', 'sum')).reset_index())
hm['halt_frac'] = hm['n_halt'] / hm['n_days']

px = pd.read_parquet('prices.parquet')
px['date'] = pd.to_datetime(px['date'])
px = px.sort_values(['code', 'date']).reset_index(drop=True)
px['ym'] = pd.PeriodIndex(px['date'], freq='M')
px['p_prev'] = px.groupby('code')['price'].shift(1)
px['frozen'] = (px['price'].notna() & px['p_prev'].notna() &
                ((px['price'] - px['p_prev']).abs() < 1e-9)).astype(int)
DAILY_CODES = set(d['code'].unique())
px['no_daily'] = (~px['code'].isin(DAILY_CODES)).astype(int)

hm = px[['code', 'ym', 'ret', 'frozen', 'no_daily']].merge(hm, on=['code', 'ym'], how='outer')
for c, v in [('halt_frac', 0.0), ('n_days', 0), ('n_halt', 0), ('n_ext', 0), ('frozen', 0)]:
    hm[c] = hm[c].fillna(v)
hm['no_daily'] = hm['no_daily'].fillna(1).astype(int)
hm['frozen'] = hm['frozen'].astype(int)

hm['by_daily'] = (hm['halt_frac'] >= FULL_HALT).astype(int)
hm['by_price'] = ((hm['no_daily'] == 1) & (hm['frozen'] == 1)).astype(int)
hm['reopen'] = (((hm['by_daily'] == 1) | (hm['by_price'] == 1)) &
                hm['ret'].notna() & hm['ret'].abs().gt(1e-9)).astype(int)
hm['full_halt'] = (((hm['by_daily'] == 1) | (hm['by_price'] == 1)) &
                   (hm['reopen'] == 0)).astype(int)
hm['src'] = np.select(
    [hm['reopen'] == 1, hm['by_daily'] == 1, hm['by_price'] == 1],
    ['재개월(제외)', '일별', '가격(일별없음)'], default='정지아님')

hm[['code', 'ym', 'n_days', 'n_halt', 'n_ext', 'halt_frac', 'frozen', 'no_daily',
    'by_daily', 'by_price', 'reopen', 'full_halt', 'src']].to_parquet('halt_month.parquet')

sens = []
for N in SENS_N:
    hh = (d['is0'] == 1) & (d['runlen_yr'] >= N)
    a = (pd.DataFrame({'code': d['code'], 'ym': d['ym'], 'h': hh.astype(int)})
         .groupby(['code', 'ym']).agg(n=('h', 'size'), k=('h', 'sum')).reset_index())
    a['f'] = a['k'] / a['n']
    sens.append({'기준일수': N, '정지일': int(hh.sum()),
                 '전체대비%': round(hh.mean() * 100, 2),
                 '완전정지월': int((a['f'] >= FULL_HALT).sum()),
                 '부분정지월': int(((a['f'] > 0) & (a['f'] < FULL_HALT)).sum()),
                 '정지종목': int(d.loc[hh, 'code'].nunique())})
sens = pd.DataFrame(sens).set_index('기준일수')
display(sens)

src = hm.groupby('src').agg(월수=('code', 'size'), 종목=('code', 'nunique'))
src.index.name = '판정출처'
display(src)

ext = d[d['extreme'] == 1][['code', 'date', 'dret']].sort_values('dret', ascending=False)
ext.index = range(1, len(ext) + 1)
display(ext)

yrd = (d.groupby('year').agg(거래일=('halt', 'size'), 정지일=('halt', 'sum'),
                             극단값=('extreme', 'sum')))
yrd['정지%'] = (yrd['정지일'] / yrd['거래일'] * 100).round(2)
yrd.index.name = '연도'
display(yrd)

ov = pd.DataFrame({'값': [
    HALT_N, FULL_HALT, EXTREME,
    len(d), int(d['halt'].sum()), int(d['extreme'].sum()),
    d['code'].nunique(), int(d.loc[d['halt'] == 1, 'code'].nunique()),
    len(hm), int(hm['full_halt'].sum()),
    int(hm.loc[hm['by_daily'] == 1, 'full_halt'].sum()),
    int(hm.loc[hm['by_price'] == 1, 'full_halt'].sum()),
    int(hm['reopen'].sum()),
    int(hm.loc[hm['no_daily'] == 1, 'code'].nunique())]},
    index=pd.Index(['정지 기준일수', '완전정지 임계', '극단값 임계(%)',
                    '일별 행', '  정지일', '  극단값',
                    '일별 종목', '  정지 경험 종목',
                    '월별 행', '완전정지 월', '  일별 판정', '  가격 판정',
                    '재개월(제외)', '일별데이터 없는 종목'], name='항목'))
display(ov)

,정지일,전체대비%,완전정지월,부분정지월,정지종목
기준일수,,,,,
5,51829,1.20,1683,2393,628
8,48954,1.13,1681,1821,560
10,48033,1.11,1679,1675,535
15,43160,1.00,1648,1078,394


,월수,종목
판정출처,,
가격(일별없음),268,31
일별,1640,235
재개월(제외),42,39
정지아님,351183,1107


,code,date,dret
1,A008080,2013-09-11,6699900.00
2,A018570,2002-04-29,863.41
3,A004230,2009-04-29,500.00
4,A005560,2003-04-15,425.25
5,A004550,2012-05-14,400.00
6,A007050,2009-04-08,333.33
7,A001920,2002-03-29,200.00
8,A005060,2002-04-09,200.00
9,A016160,2011-04-20,182.05
10,A008020,2016-05-11,153.66


,거래일,정지일,극단값,정지%
연도,,,,
1999,621,0,0,0.00
2000,148709,880,0,0.59
2001,151468,1499,0,0.99
2002,148501,1667,4,1.12
2003,150356,1812,1,1.21
2004,149346,1480,0,0.99
2005,148257,2002,0,1.35
2006,149275,1815,0,1.22
2007,150671,1379,0,0.92


,값
항목,
정지 기준일수,8.00
완전정지 임계,0.95
극단값 임계(%),100.00
일별 행,4323086.00
정지일,48954.00
극단값,15.00
일별 종목,953.00
정지 경험 종목,560.00
월별 행,353133.00


In [6]:
# 4

DRET_CAP = 100.0
MIN_M12, MIN_D12, MIN_D03 = 3, 20, 5
YR0, YR1 = 2001, 2025

hd = pd.read_parquet('halt_daily.parquet'); hd['date'] = pd.to_datetime(hd['date'])
hm = pd.read_parquet('halt_month.parquet')
px = pd.read_parquet('prices.parquet'); px['date'] = pd.to_datetime(px['date'])
px['ym'] = pd.PeriodIndex(px['date'], freq='M'); px['year'] = px['date'].dt.year
mkt = pd.read_parquet('market.parquet'); mkt['date'] = pd.to_datetime(mkt['date'])
mkt['ym'] = pd.PeriodIndex(mkt['date'], freq='M')
mkt_m = (mkt.assign(g=1 + mkt['kret'] / 100).groupby('ym')['g'].prod().sub(1)
         .rename('mret').reset_index())
mk_d = mkt[['date', 'kret']].dropna().assign(mret_d=lambda x: x['kret'] / 100)[['date', 'mret_d']]

dd = hd[(hd['halt'] == 0) & (hd['extreme'] == 0) & hd['dret'].notna()].copy()
dd = dd[(dd['dret'].abs() <= DRET_CAP) & (dd['year'] >= YR0) & (dd['year'] <= YR1)]
dd['q'] = pd.PeriodIndex(dd['date'], freq='Q')
dd = dd.merge(mk_d, on='date', how='inner')
dd['yv'] = dd['dret'] / 100
dd['key'] = dd['code'] + '_' + dd['year'].astype(str)

pm = px.merge(hm[['code', 'ym', 'full_halt']], on=['code', 'ym'], how='left')
pm['full_halt'] = pm['full_halt'].fillna(0)
pm = pm[(pm['full_halt'] == 0) & pm['ret'].notna() &
        (pm['year'] >= YR0) & (pm['year'] <= YR1)]
pm = pm.merge(mkt_m, on='ym', how='inner')
pm['yv'] = pm['ret'] / 100
pm['key'] = pm['code'] + '_' + pm['year'].astype(str)

def sig_group(df, xcol, minobs):
    uniq, gid = np.unique(df['key'].values, return_inverse=True)
    G = len(uniq)
    y = df['yv'].values.astype('float64')
    n = np.bincount(gid, minlength=G).astype('float64')
    sy = np.bincount(gid, weights=y, minlength=G)
    syy = np.bincount(gid, weights=y * y, minlength=G)
    if xcol is None:
        with np.errstate(divide='ignore', invalid='ignore'):
            sd = np.sqrt((syy - sy * sy / n) / (n - 1))
        ok = (n >= max(minobs, 2)) & (sd > 0)
    else:
        x = df[xcol].values.astype('float64')
        sx = np.bincount(gid, weights=x, minlength=G)
        sxx = np.bincount(gid, weights=x * x, minlength=G)
        sxy = np.bincount(gid, weights=x * y, minlength=G)
        det = n * sxx - sx * sx
        with np.errstate(divide='ignore', invalid='ignore'):
            b0 = (sxx * sy - sx * sxy) / det
            b1 = (n * sxy - sx * sy) / det
            ssr = syy - b0 * sy - b1 * sxy
            sd = np.sqrt(ssr / (n - 2))
        ok = (n >= max(minobs, 3)) & (det > 0) & (ssr > 0)
    o = pd.DataFrame({'key': uniq, 'n': n.astype(int), 'sig': np.where(ok, sd, np.nan)})
    o['code'] = [k.rsplit('_', 1)[0] for k in o['key']]
    o['year'] = [int(k.rsplit('_', 1)[1]) for k in o['key']]
    return o[['code', 'year', 'n', 'sig']]

SPEC = {
    'SIGMA_m12':  (pm,                              'mret',   MIN_M12),
    'SIGMA_d12':  (dd,                              'mret_d', MIN_D12),
    'SIGMA_d03':  (dd[dd['q'].dt.quarter == 4],     'mret_d', MIN_D03),
    'SIGMA_d12t': (dd,                              None,     MIN_D12),
}

sigma = None
for name, (src, xc, mo) in SPEC.items():
    o = sig_group(src, xc, mo).rename(
        columns={'sig': name, 'n': 'n_' + name.replace('SIGMA_', '')})
    sigma = o if sigma is None else sigma.merge(o, on=['code', 'year'], how='outer')

NCOL = [c for c in sigma.columns if c.startswith('n_')]
sigma[NCOL] = sigma[NCOL].fillna(0).astype(int)
sigma = sigma.sort_values(['code', 'year']).reset_index(drop=True)
sigma.to_parquet('sigma.parquet')

SCOL = list(SPEC)
spec = pd.DataFrame({
    '창': ['월별 12개월', '일별 12개월', '일별 4분기', '일별 12개월'],
    '벤치마크': ['시장모형 잔차', '시장모형 잔차', '시장모형 잔차', '없음(총변동성)'],
    '최소 관측': [MIN_M12, MIN_D12, MIN_D03, MIN_D12],
    '유효 종목-연도': [int(sigma[c].notna().sum()) for c in SCOL],
    '결측': [int(sigma[c].isna().sum()) for c in SCOL],
}, index=pd.Index(SCOL, name='정의'))
display(spec)

desc = sigma[SCOL].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).T
desc.index.name = '정의'
display(desc.round(4))

corr = sigma[SCOL].corr()
corr.index.name = '정의'
display(corr.round(3))

corr_s = sigma[SCOL].corr(method='spearman')
corr_s.index.name = '정의'
display(corr_s.round(3))

byyr = sigma.groupby('year')[SCOL].apply(lambda g: g.notna().sum())
byyr['종목'] = sigma.groupby('year')['code'].nunique()
byyr.index.name = '연도'
display(byyr)

ov = pd.DataFrame({'값': [
    DRET_CAP, MIN_M12, MIN_D12, MIN_D03,
    len(sigma), sigma['code'].nunique(),
    f"{int(sigma['year'].min())}~{int(sigma['year'].max())}",
    int(sigma[SCOL].notna().all(axis=1).sum())]},
    index=pd.Index(['dret 상한(%)', '최소 관측 m12', '최소 관측 d12', '최소 관측 d03',
                    '종목-연도 행', '종목', '기간', '4종 모두 유효'], name='항목'))
display(ov)

,창,벤치마크,최소 관측,유효 종목-연도,결측
정의,,,,,
SIGMA_m12,월별 12개월,시장모형 잔차,3,17125,166
SIGMA_d12,일별 12개월,시장모형 잔차,20,16729,562
SIGMA_d03,일별 4분기,시장모형 잔차,5,16571,720
SIGMA_d12t,일별 12개월,없음(총변동성),20,16729,562


,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
정의,,,,,,,,,,,,
SIGMA_m12,17125.0,0.1236,0.1238,0.0005,0.0268,0.0410,0.0692,0.0983,0.1438,0.2772,0.5214,9.0407
SIGMA_d12,16729.0,0.0288,0.0160,0.0022,0.0091,0.0127,0.0192,0.0255,0.0341,0.0553,0.0779,0.2472
SIGMA_d03,16571.0,0.0267,0.0167,0.0012,0.0074,0.0106,0.0166,0.0226,0.0322,0.0567,0.0798,0.5527
SIGMA_d12t,16729.0,0.0307,0.0161,0.0022,0.0096,0.0135,0.0206,0.0275,0.0369,0.0571,0.0791,0.2465


,SIGMA_m12,SIGMA_d12,SIGMA_d03,SIGMA_d12t
정의,,,,
SIGMA_m12,1.000,0.620,0.572,0.611
SIGMA_d12,0.620,1.000,0.778,0.990
SIGMA_d03,0.572,0.778,1.000,0.773
SIGMA_d12t,0.611,0.990,0.773,1.000


,SIGMA_m12,SIGMA_d12,SIGMA_d03,SIGMA_d12t
정의,,,,
SIGMA_m12,1.000,0.811,0.673,0.799
SIGMA_d12,0.811,1.000,0.786,0.983
SIGMA_d03,0.673,0.786,1.000,0.777
SIGMA_d12t,0.799,0.983,0.777,1.000


,SIGMA_m12,SIGMA_d12,SIGMA_d03,SIGMA_d12t,종목
연도,,,,,
2001,632,630,619,630,641
2002,635,629,611,629,644
2003,627,615,603,615,632
2004,622,607,598,607,630
2005,618,610,596,610,632
2006,624,608,607,608,629
2007,636,621,619,621,646
2008,656,630,627,630,658
2009,666,639,630,639,673


,값
항목,
dret 상한(%),100.0
최소 관측 m12,3
최소 관측 d12,20
최소 관측 d03,5
종목-연도 행,17291
종목,983
기간,2001~2025
4종 모두 유효,16493


In [7]:
# 5

MIN_MONTHS = 12
FULL_HALT = 0.95
YR0, YR1 = 2001, 2025

px = pd.read_parquet('prices.parquet'); px['date'] = pd.to_datetime(px['date'])
px['ym'] = pd.PeriodIndex(px['date'], freq='M'); px['year'] = px['date'].dt.year
hm = pd.read_parquet('halt_month.parquet')
mkt = pd.read_parquet('market.parquet')
mkt['ym'] = pd.PeriodIndex(mkt['date'], freq='M')
mkt_m = (mkt.assign(g=1 + mkt['kret'] / 100).groupby('ym')['g'].prod().sub(1)
         .rename('mret').reset_index())
mkt_m['year'] = mkt_m['ym'].dt.year

px = px.merge(hm[['code', 'ym', 'n_days', 'n_halt', 'full_halt']],
              on=['code', 'ym'], how='left')
for c in ['n_days', 'n_halt', 'full_halt']:
    px[c] = px[c].fillna(0)
px = px[(px['year'] >= YR0) & (px['year'] <= YR1)].copy()
px['mret_i'] = px['ret'] / 100

hy = px.groupby(['code', 'year']).agg(
    d_tot=('n_days', 'sum'), d_halt=('n_halt', 'sum'),
    m_tot=('ym', 'size'), m_halt=('full_halt', 'sum')).reset_index()
hy['halt_frac_y'] = np.where(hy['d_tot'] > 0, hy['d_halt'] / hy['d_tot'],
                             hy['m_halt'] / hy['m_tot'])
hy['full_year'] = (hy['m_halt'] / hy['m_tot'] >= FULL_HALT).astype(int)
hy = hy[['code', 'year', 'halt_frac_y', 'full_year']]

r = px[px['mret_i'].notna()].copy()
r['g'] = 1 + r['mret_i']
fy = r.groupby(['code', 'year']).agg(g=('g', 'prod'), n_ret=('mret_i', 'count')).reset_index()
m_ann = (mkt_m.assign(g=1 + mkt_m['mret']).groupby('year')['g'].prod().sub(1)
         .rename('m_ann').reset_index())
fy = fy.merge(m_ann, on='year', how='left')
fy['r_ann'] = fy['g'] - 1
fy['EXRET'] = np.where(fy['n_ret'] >= MIN_MONTHS, fy['r_ann'] - fy['m_ann'], np.nan)
fy = fy.merge(hy, on=['code', 'year'], how='left')
fy.loc[fy['full_year'] == 1, 'EXRET'] = np.nan
exret = fy[['code', 'year', 'EXRET', 'r_ann', 'm_ann', 'n_ret']]

dec = px[(px['ym'].dt.month == 12) & px['mktcap'].notna()][['code', 'year', 'mktcap']]
tot = dec.groupby('year')['mktcap'].sum().rename('mktcap_tot')
rsize = dec.join(tot, on='year')
rsize['RSIZE'] = np.log(rsize['mktcap'] / rsize['mktcap_tot'])
rsize = rsize[['code', 'year', 'RSIZE', 'mktcap']]

mv = (exret.merge(rsize, on=['code', 'year'], how='outer')
      .merge(hy, on=['code', 'year'], how='left'))
mv[['halt_frac_y', 'full_year']] = mv[['halt_frac_y', 'full_year']].fillna(0)
mv['full_year'] = mv['full_year'].astype(int)
mv = mv.sort_values(['code', 'year']).reset_index(drop=True)
mv.to_parquet('mktvars.parquet')

spec = pd.DataFrame({
    '정의': ['연간 복리수익률 − 시장 연간수익률', '12월 시총의 시장 전체 대비 로그 비중'],
    '요건': [f'월별 수익률 {MIN_MONTHS}개월 완비', '12월 시총 존재'],
    '정지 처리': [f'연 완전정지({FULL_HALT:.0%}+)는 결측', '동결 시총 유지'],
    '유효': [int(mv['EXRET'].notna().sum()), int(mv['RSIZE'].notna().sum())],
    '결측': [int(mv['EXRET'].isna().sum()), int(mv['RSIZE'].isna().sum())]},
    index=pd.Index(['EXRET', 'RSIZE'], name='변수'))
display(spec)

desc = mv[['EXRET', 'RSIZE', 'r_ann', 'm_ann', 'n_ret']].describe(
    percentiles=[.01, .05, .25, .5, .75, .95, .99]).T
desc.index.name = '변수'
display(desc.round(4))

rows = []
for lab, m in [('정지 없음', mv['halt_frac_y'] == 0),
               ('부분 정지', (mv['halt_frac_y'] > 0) & (mv['full_year'] == 0)),
               ('연 완전정지', mv['full_year'] == 1)]:
    s = mv[m]
    rows.append({'구분': lab, '종목-연도': len(s),
                 'EXRET 유효': int(s['EXRET'].notna().sum()),
                 'EXRET 결측 %': round(s['EXRET'].isna().mean() * 100, 1),
                 'EXRET 중앙': round(s['EXRET'].median(), 4),
                 'RSIZE 유효': int(s['RSIZE'].notna().sum()),
                 'RSIZE 중앙': round(s['RSIZE'].median(), 3)})
halt = pd.DataFrame(rows).set_index('구분')
display(halt)

byyr = mv.groupby('year').agg(
    종목=('code', 'nunique'),
    EXRET=('EXRET', lambda s: int(s.notna().sum())),
    RSIZE=('RSIZE', lambda s: int(s.notna().sum())))
byyr['시장수익률'] = mkt_m.groupby('year')['mret'].apply(
    lambda s: np.nan).reindex(byyr.index)
byyr['시장수익률'] = m_ann.set_index('year')['m_ann'].reindex(byyr.index).round(4)
byyr['EXRET 중앙'] = mv.groupby('year')['EXRET'].median().round(4)
byyr.index.name = '연도'
display(byyr)

ov = pd.DataFrame({'값': [
    MIN_MONTHS, FULL_HALT, len(mv), mv['code'].nunique(),
    f"{int(mv['year'].min())}~{int(mv['year'].max())}",
    int((mv['EXRET'].notna() & mv['RSIZE'].notna()).sum())]},
    index=pd.Index(['EXRET 최소 개월', '연 완전정지 임계', '종목-연도 행', '종목',
                    '기간', '두 변수 모두 유효'], name='항목'))
display(ov)

,정의,요건,정지 처리,유효,결측
변수,,,,,
EXRET,연간 복리수익률 − 시장 연간수익률,월별 수익률 12개월 완비,연 완전정지(95%+)는 결측,16745,599
RSIZE,12월 시총의 시장 전체 대비 로그 비중,12월 시총 존재,동결 시총 유지,17143,201


,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
변수,,,,,,,,,,,,
EXRET,16745.0,0.0419,0.7115,-1.6781,-1.0042,-0.6896,-0.2837,-0.0568,0.2159,1.0292,2.3931,37.0909
RSIZE,17143.0,-8.4892,1.6536,-15.4961,-11.2758,-10.6448,-9.6642,-8.8247,-7.5958,-5.2086,-3.8516,-1.2916
r_ann,17336.0,0.1623,0.7379,-0.9930,-0.7616,-0.5204,-0.1933,0.0199,0.3261,1.2602,2.6722,37.6589
m_ann,17336.0,0.1289,0.2780,-0.4036,-0.4036,-0.2593,-0.0604,0.0807,0.3031,0.5680,0.8621,0.8621
n_ret,17336.0,11.7851,1.3052,1.0000,3.0000,12.0000,12.0000,12.0000,12.0000,12.0000,12.0000,12.0000


,종목-연도,EXRET 유효,EXRET 결측 %,EXRET 중앙,RSIZE 유효,RSIZE 중앙
구분,,,,,,
정지 없음,16224,15833,2.4,-0.0493,16181,-8.795
부분 정지,1074,912,15.1,-0.2470,916,-9.341
연 완전정지,46,0,100.0,NaN,46,-9.959


,종목,EXRET,RSIZE,시장수익률,EXRET 중앙
연도,,,,,
2001,640,609,626,0.4469,-0.1338
2002,643,604,623,-0.0927,-0.0222
2003,632,611,619,0.3031,-0.2341
2004,630,605,616,0.0807,-0.0233
2005,631,600,616,0.5680,0.2724
2006,633,608,629,0.0462,-0.0209
2007,646,621,637,0.3189,-0.0512
2008,658,636,656,-0.4036,-0.0431
2009,675,639,659,0.5203,-0.0865


,값
항목,
EXRET 최소 개월,12
연 완전정지 임계,0.95
종목-연도 행,17344
종목,982
기간,2001~2025
두 변수 모두 유효,16745


In [8]:
# 6

YR0, YR1 = 2001, 2025

fin = pd.read_parquet('financials.parquet')
fin['date'] = pd.to_datetime(fin['date'])
fin = fin.sort_values(['code', 'date']).reset_index(drop=True)

pq = pd.PeriodIndex(fin['date'], freq='Q')
fin['qtr'] = pq
fin['qidx'] = pq.year * 4 + (pq.quarter - 1)

g = fin.groupby('code', sort=False)
ok4 = (fin['qidx'] - g['qidx'].shift(3)) == 3
fin['ibq_ttm'] = np.where(
    ok4, g['ibq'].apply(lambda x: x.rolling(4, min_periods=4).sum())
    .reset_index(level=0, drop=True), np.nan)

acc = fin[fin['qtr'].dt.quarter == 2][
    ['code', 'qtr', 'ibq_ttm', 'ltq', 'atq', 'seqq']].copy()
acc['year'] = acc['qtr'].dt.year
den = acc['atq'].where(acc['atq'] > 0)
acc['NITA'] = acc['ibq_ttm'] / den
acc['TLTA'] = acc['ltq'] / den
acc['atq_pos'] = acc['atq'].gt(0).astype(int)
acc = acc[(acc['year'] >= YR0) & (acc['year'] <= YR1)]
acc = acc[['code', 'year', 'NITA', 'TLTA', 'ibq_ttm', 'ltq', 'atq', 'seqq', 'atq_pos']]
acc = acc.sort_values(['code', 'year']).reset_index(drop=True)
acc.to_parquet('accvars.parquet')

spec = pd.DataFrame({
    '정의': ['TTM 순이익 / 총자산', '총부채 / 총자산'],
    '분자': ['ibq 4분기 합 (연속 4분기 요건)', 'ltq'],
    '분모': ['atq (>0 조건)', 'atq (>0 조건)'],
    '기준 시점': ['2분기 (6월말)', '2분기 (6월말)'],
    '유효': [int(acc['NITA'].notna().sum()), int(acc['TLTA'].notna().sum())],
    '결측': [int(acc['NITA'].isna().sum()), int(acc['TLTA'].isna().sum())]},
    index=pd.Index(['NITA', 'TLTA'], name='변수'))
display(spec)

desc = acc[['NITA', 'TLTA']].describe(
    percentiles=[.01, .05, .25, .5, .75, .95, .99]).T
desc.index.name = '변수'
display(desc.round(4))

na = pd.DataFrame({
    '건수': [len(acc), int(acc['atq'].isna().sum()), int(acc['atq'].le(0).sum()),
            int(acc['ibq_ttm'].isna().sum()), int(acc['ltq'].isna().sum()),
            int(acc['NITA'].isna().sum()), int(acc['TLTA'].isna().sum())]},
    index=pd.Index(['전체 종목-연도', 'atq 결측', 'atq ≤ 0', 'ibq_ttm 결측 (4분기 미충족)',
                    'ltq 결측', 'NITA 결측', 'TLTA 결측'], name='결측 원인'))
na['비율 %'] = (na['건수'] / len(acc) * 100).round(2)
display(na)

byyr = acc.groupby('year').agg(
    NITA=('NITA', lambda s: int(s.notna().sum())),
    TLTA=('TLTA', lambda s: int(s.notna().sum())))
byyr['NITA 중앙'] = acc.groupby('year')['NITA'].median().round(4)
byyr['TLTA 중앙'] = acc.groupby('year')['TLTA'].median().round(4)
byyr['TLTA>1'] = acc.groupby('year')['TLTA'].apply(lambda s: int((s > 1).sum()))
byyr['TLTA>1 %'] = (byyr['TLTA>1'] / byyr['TLTA'] * 100).round(2)
byyr.index.name = '연도'
display(byyr)

ov = pd.DataFrame({'값': [
    len(acc), acc['code'].nunique(),
    f"{int(acc['year'].min())}~{int(acc['year'].max())}",
    int((acc['NITA'].notna() & acc['TLTA'].notna()).sum()),
    int(acc['TLTA'].gt(1).sum())]},
    index=pd.Index(['종목-연도 행', '종목', '기간', '두 변수 모두 유효',
                'TLTA > 1 (완전자본잠식)'], name='항목'))
display(ov)

,정의,분자,분모,기준 시점,유효,결측
변수,,,,,,
NITA,TTM 순이익 / 총자산,ibq 4분기 합 (연속 4분기 요건),atq (>0 조건),2분기 (6월말),16965,10710
TLTA,총부채 / 총자산,ltq,atq (>0 조건),2분기 (6월말),17386,10289


,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
변수,,,,,,,,,,,,
NITA,16965.0,0.0108,0.3455,-21.3847,-0.5263,-0.1476,-0.0029,0.0272,0.0598,0.1265,0.2306,21.9694
TLTA,17386.0,0.4793,0.3566,0.0001,0.0650,0.1357,0.3108,0.4750,0.6201,0.8138,0.9957,29.8678


,건수,비율 %
결측 원인,,
전체 종목-연도,27675,100.00
atq 결측,10289,37.18
atq ≤ 0,0,0.00
ibq_ttm 결측 (4분기 미충족),10710,38.70
ltq 결측,10289,37.18
NITA 결측,10710,38.70
TLTA 결측,10289,37.18


,NITA,TLTA,NITA 중앙,TLTA 중앙,TLTA>1,TLTA>1 %
연도,,,,,,
2001,594,649,0.0121,0.5615,66,10.17
2002,611,641,0.0342,0.5162,18,2.81
2003,626,649,0.0316,0.4983,17,2.62
2004,622,640,0.0364,0.4699,8,1.25
2005,621,640,0.0380,0.4533,3,0.47
2006,631,647,0.0394,0.4554,1,0.15
2007,637,654,0.0391,0.4607,0,0.00
2008,652,673,0.0341,0.4738,2,0.30
2009,656,678,0.0227,0.4598,5,0.74


,값
항목,
종목-연도 행,27675
종목,1107
기간,2001~2025
두 변수 모두 유효,16965
TLTA > 1 (완전자본잠식),162


In [9]:
# 7

FILL_LIMIT = 2
SUB_SCOPE = 'halt'
SVARS = ['SIGMA_m12', 'SIGMA_d12', 'SIGMA_d03', 'SIGMA_d12t']
MVARS = ['EXRET', 'RSIZE']
AVARS = ['NITA', 'TLTA']
ANN = {'SIGMA_m12': np.sqrt(12), 'SIGMA_d12': np.sqrt(252),
       'SIGMA_d03': np.sqrt(252), 'SIGMA_d12t': np.sqrt(252)}
BASE = AVARS + MVARS
AVAR = [s + '_a' for s in SVARS]
ALLV = BASE + SVARS + AVAR + ['SIGMA_d12_fill', 'SIGMA_d12_fill_a']

sg = pd.read_parquet('sigma.parquet')
mv = pd.read_parquet('mktvars.parquet')
ac = pd.read_parquet('accvars.parquet')

for s in SVARS:
    sg[s + '_a'] = sg[s] * ANN[s]

X = (sg[['code', 'year'] + SVARS + AVAR]
     .merge(mv[['code', 'year'] + MVARS + ['halt_frac_y', 'full_year']],
            on=['code', 'year'], how='outer')
     .merge(ac[['code', 'year'] + AVARS], on=['code', 'year'], how='outer'))

full = (X[['code']].drop_duplicates()
        .merge(pd.DataFrame({'year': range(int(X['year'].min()), int(X['year'].max()) + 1)}),
               how='cross')
        .merge(X, on=['code', 'year'], how='left')
        .sort_values(['code', 'year']).reset_index(drop=True))

VFILL = BASE + SVARS + AVAR
gg = full.groupby('code', sort=False)
for c in VFILL:
    filled = gg[c].ffill(limit=FILL_LIMIT)
    full[c + '_f'] = filled
    last = full['year'].where(full[c].notna()).groupby(full['code']).ffill()
    full[c + '_fillage'] = np.where(full[c].notna(), 0,
                                    np.where(filled.notna(), full['year'] - last, np.nan))

full['SIGMA_d12_fill'] = full['SIGMA_d12']
full['SIGMA_d12_fill_a'] = full['SIGMA_d12_a']
full['SIGMA_d12_fill_f'] = full['SIGMA_d12_f']
full['SIGMA_d12_fill_a_f'] = full['SIGMA_d12_a_f']
full['SIGMA_d12_fill_fillage'] = full['SIGMA_d12_fillage']
full['SIGMA_d12_fill_a_fillage'] = full['SIGMA_d12_a_fillage']
need = full['SIGMA_d12_a_f'].isna() & full['SIGMA_m12_a_f'].notna()
full['d12_from_m12'] = need.astype(int)
full.loc[need, 'SIGMA_d12_fill_a_f'] = full.loc[need, 'SIGMA_m12_a_f']
full.loc[need, 'SIGMA_d12_fill_f'] = full.loc[need, 'SIGMA_m12_a_f'] / ANN['SIGMA_d12']

SUBC = SVARS + AVAR + ['SIGMA_d12_fill', 'SIGMA_d12_fill_a']
for c in SUBC:
    full[c + '_sub'] = 0
    if SUB_SCOPE is None:
        continue
    nd = full[c + '_f'].isna()
    if SUB_SCOPE == 'halt':
        nd = nd & (full['halt_frac_y'] > 0)
    xm = full.groupby('year')[c + '_f'].transform('mean')
    hit = nd & xm.notna()
    full.loc[hit, c + '_sub'] = 1
    full.loc[hit, c + '_f'] = xm[hit]

full['obs_year'] = full['year'] + 1
X2 = full.drop(columns='year')

panel = (pd.read_parquet('riskset.parquet')
         .merge(X2, on=['code', 'obs_year'], how='left'))

FV = [c + '_f' for c in ALLV]
AG = [c + '_fillage' for c in ALLV]
SB = [c + '_sub' for c in SUBC]
panel['n_filled'] = (panel[[c + '_fillage' for c in BASE]] > 0).sum(axis=1)
panel['any_filled'] = (panel['n_filled'] > 0).astype(int)
panel['halt_frac_y'] = panel['halt_frac_y'].fillna(0.0)
panel['full_year'] = panel['full_year'].fillna(0).astype(int)
panel['halted'] = (panel['halt_frac_y'] > 0).astype(int)
panel['d12_from_m12'] = panel['d12_from_m12'].fillna(0).astype(int)

comp_f = panel[[c + '_f' for c in BASE + ['SIGMA_d12_a']]].notna().all(axis=1)
lo = panel.loc[comp_f, FV].quantile(0.01)
hi = panel.loc[comp_f, FV].quantile(0.99)
for c in ALLV:
    panel[c + '_w'] = panel[c + '_f'].clip(lo[c + '_f'], hi[c + '_f'])

KEEP = (['code', 'obs_year', 'y', 'censored', 'n_filled', 'any_filled',
         'halt_frac_y', 'halted', 'full_year', 'd12_from_m12']
        + ALLV + FV + AG + SB + [c + '_w' for c in ALLV])
panel = panel[[c for c in KEEP if c in panel.columns]]
panel.to_parquet('shumway_panel.parquet')

ov = pd.DataFrame({'값': [
    len(panel), panel['code'].nunique(),
    f"{int(panel['obs_year'].min())}~{int(panel['obs_year'].max())}",
    int(panel['y'].sum()), int(panel['censored'].sum()),
    FILL_LIMIT, str(SUB_SCOPE), int(panel['d12_from_m12'].sum())]},
    index=pd.Index(['패널 행', '종목', '기간', '사건', '절단',
                    'ffill 한도(년)', '횡단면대체 범위', 'd12←m12 보완'], name='항목'))
display(ov)

def path(pfx):
    return np.select(
        [panel[pfx].notna(),
         panel.get(pfx + '_fillage', pd.Series(np.nan, index=panel.index)) > 0,
         panel.get('d12_from_m12', pd.Series(0, index=panel.index)).eq(1) & ('fill' in pfx),
         panel[pfx + '_sub'] == 1],
        ['실측', 'ffill', 'm12 보완', '횡단면평균'], default='결측')

rows = []
for pfx in ['SIGMA_m12', 'SIGMA_d12', 'SIGMA_d12_fill', 'SIGMA_d03', 'SIGMA_d12t']:
    s = path(pfx)
    for k in ['실측', 'ffill', 'm12 보완', '횡단면평균', '결측']:
        m = s == k
        if not m.sum(): continue
        rows.append({'정의': pfx, '경로': k, '행': int(m.sum()),
                     '사건': int(panel.loc[m, 'y'].sum()),
                     'SIGMA 중앙': round(panel.loc[m, pfx + '_a_f'].median(), 4),
                     '사건 SIGMA': round(panel.loc[m & panel['y'].eq(1), pfx + '_a_f'].median(), 4)
                     if (m & panel['y'].eq(1)).sum() else np.nan})
src = pd.DataFrame(rows).set_index(['정의', '경로'])
display(src)

rows = []
for lab, c in [('회계+시장 4변수', None), ('+ m12', 'SIGMA_m12'), ('+ d12', 'SIGMA_d12'),
               ('+ d12(보완)', 'SIGMA_d12_fill'), ('+ d03', 'SIGMA_d03'),
               ('+ d12t', 'SIGMA_d12t')]:
    cols = BASE + ([c + '_a'] if c else [])
    raw = panel[BASE + ([c] if c else [])].notna().all(axis=1)
    fil = panel[[x + '_f' for x in cols]].notna().all(axis=1)
    rows.append({'모형 표본': lab,
                 '대체 전 행': int(raw.sum()), '대체 전 사건': int(panel.loc[raw, 'y'].sum()),
                 '대체 후 행': int(fil.sum()), '대체 후 사건': int(panel.loc[fil, 'y'].sum())})
samp = pd.DataFrame(rows).set_index('모형 표본')
display(samp)

rows = []
for c in SVARS + ['SIGMA_d12_fill']:
    rows.append({'정의': c,
                 '원값 중앙': round(panel[c].median(), 4),
                 '연율 중앙': round(panel[c + '_a'].median(), 4),
                 '윈저 하한': round(panel[c + '_a_w'].min(), 4),
                 '윈저 상한': round(panel[c + '_a_w'].max(), 4)})
sig = pd.DataFrame(rows).set_index('정의')
display(sig)

z = panel['halt_frac_y']
ht = pd.DataFrame({
    '행': [int(z.eq(0).sum()), int(((z > 0) & (z < 0.25)).sum()),
          int(((z >= 0.25) & (z < 0.95)).sum()), int(z.ge(0.95).sum())],
    '사건': [int(panel.loc[z.eq(0), 'y'].sum()),
            int(panel.loc[(z > 0) & (z < 0.25), 'y'].sum()),
            int(panel.loc[(z >= 0.25) & (z < 0.95), 'y'].sum()),
            int(panel.loc[z.ge(0.95), 'y'].sum())]},
    index=pd.Index(['0%', '0~25%', '25~95%', '95%+'], name='연간 정지율'))
ht['사건률 %'] = (ht['사건'] / ht['행'] * 100).round(2)
display(ht)

cm = panel[[c + '_f' for c in BASE + ['SIGMA_d12_a']]].notna().all(axis=1)
byyr = panel[cm].groupby('obs_year').agg(표본=('y', 'size'), 사건=('y', 'sum'))
byyr['사건률 %'] = (byyr['사건'] / byyr['표본'] * 100).round(2)
byyr.loc['합계'] = [byyr['표본'].sum(), byyr['사건'].sum(),
                  round(byyr['사건'].sum() / byyr['표본'].sum() * 100, 2)]
byyr.index.name = '관측연도'
display(byyr)

,값
항목,
패널 행,16657
종목,962
기간,2002~2025
사건,102
절단,80
ffill 한도(년),2
횡단면대체 범위,halt
d12←m12 보완,419


행  사건  SIGMA 중앙  사건 SIGMA
정의             경로                                   
SIGMA_m12      실측      16171  88    0.3413    0.8070
               ffill      45  13    0.5988    0.7387
               횡단면평균       4   1    0.3546    0.3544
               결측        437   0       NaN       NaN
SIGMA_d12      실측      15801  81    0.4048    0.9448
               ffill      37   8    0.8788    0.9273
               횡단면평균      60   8    0.4537    0.4023
               결측        759   5       NaN       NaN
SIGMA_d12_fill 실측      15801  81    0.4048    0.9448
               ffill      37   8    0.8788    0.9273
               m12 보완    419  12    0.4091    0.8290
               횡단면평균       4   1    0.4043    0.4043
               결측        396   0       NaN       NaN
SIGMA_d03      실측      15779  75    0.3606    1.0157
               ffill      78  13    0.7371    0.8576
               횡단면평균      65   9    0.4155    0.4155
               결측        735   5       NaN       NaN
SIGMA_d12t     실측      15801  81    0.4381    0.9843
               ffill      37   8    0.8959    0.9229
               횡단면평균      60   8    0.4715    0.4301
               결측        759   5       NaN       NaN

,대체 전 행,대체 전 사건,대체 후 행,대체 후 사건
모형 표본,,,,
회계+시장 4변수,15831,90,15870,100
+ m12,15817,88,15870,100
+ d12,15439,81,15539,95
+ d12(보완),15439,81,15870,100
+ d03,15393,75,15539,95
+ d12t,15439,81,15539,95


,원값 중앙,연율 중앙,윈저 하한,윈저 상한
정의,,,,
SIGMA_m12,0.0985,0.3413,0.0941,1.7052
SIGMA_d12,0.0255,0.4048,0.1452,1.1264
SIGMA_d03,0.0227,0.3606,0.1174,1.2485
SIGMA_d12t,0.0276,0.4381,0.1543,1.1368
SIGMA_d12_fill,0.0255,0.4048,0.1446,1.1309


,행,사건,사건률 %
연간 정지율,,,
0%,15747,58,0.37
0~25%,767,21,2.74
25~95%,103,10,9.71
95%+,40,13,32.50


,표본,사건,사건률 %
관측연도,,,
2002,571.0,19.0,3.33
2003,594.0,5.0,0.84
2004,596.0,10.0,1.68
2005,591.0,9.0,1.52
2006,583.0,1.0,0.17
2007,594.0,4.0,0.67
2008,603.0,1.0,0.17
2009,617.0,12.0,1.94
2010,618.0,10.0,1.62


In [10]:
# 8

p = pd.read_parquet('shumway_panel.parquet')
BASE = ['NITA', 'TLTA', 'RSIZE', 'EXRET']
SDEF = {'m12': 'SIGMA_m12_a_w', 'd12': 'SIGMA_d12_a_w', 'd12_fill': 'SIGMA_d12_fill_a_w'}
MODELS = {'M1 시장': ['RSIZE', 'EXRET', 'SIGMA'],
          'M2 회계': ['NITA', 'TLTA'],
          'M3 결합': ['NITA', 'TLTA', 'RSIZE', 'EXRET', 'SIGMA'],
          'M4 SIGMA제외': ['NITA', 'TLTA', 'RSIZE', 'EXRET']}

def hazard(cols, scol):
    use = [(scol if c == 'SIGMA' else c + '_w') for c in cols]
    d = p.dropna(subset=use + ['y'])
    X = sm.add_constant(d[use].astype(float))
    X.columns = ['Intercept'] + cols
    yv = d['y'].astype(float)
    r = sm.Logit(yv, X).fit(disp=0)
    g = pd.factorize(d['code'])[0]
    rc = sm.Logit(yv, X).fit(disp=0, cov_type='cluster',
                             cov_kwds={'groups': g, 'use_correction': True})
    adj = len(d) / d['code'].nunique()
    c2 = r.tvalues ** 2
    out = pd.DataFrame({
        '계수': r.params.round(3),
        'p_보정없음': stats.chi2.sf(c2, 1).round(3),
        'p_Shumway': stats.chi2.sf(c2 / adj, 1).round(3),
        'p_클러스터': stats.chi2.sf(np.asarray(rc.tvalues) ** 2, 1).round(3)})
    meta = {'기업': d['code'].nunique(), 'firm-year': len(d), '사건': int(yv.sum()),
            '기업당연수': round(adj, 2), 'pseudo-R2': round(r.prsquared, 3)}
    return out.rename_axis('변수'), meta

coefs, metas = {}, {}
for sk, scol in SDEF.items():
    for mk, cols in MODELS.items():
        if 'SIGMA' not in cols and sk != 'd12':
            continue
        key = mk if 'SIGMA' not in cols else f'{mk} [{sk}]'
        o, m = hazard(cols, scol)
        coefs[key] = o
        metas[key] = m

fit = pd.DataFrame(metas).T
fit.index.name = '모형'
display(fit)

cf = pd.concat(coefs, names=['모형'])
display(cf)

ORDER = ['Intercept', 'NITA', 'TLTA', 'RSIZE', 'EXRET', 'SIGMA']
wide = cf['계수'].unstack('모형').reindex(ORDER)[list(coefs)]
wide.loc['사건'] = fit['사건']
wide.loc['pseudo-R2'] = fit['pseudo-R2']
wide.index.name = '변수'
display(wide)

pw = cf['p_클러스터'].unstack('모형').reindex(ORDER)[list(coefs)]
pw.index.name = '변수'
display(pw)

m3 = {k: v for k, v in coefs.items() if k.startswith('M3')}
cmp3 = pd.concat({k.split('[')[1].rstrip(']'): v for k, v in m3.items()}, names=['SIGMA 정의'])
display(cmp3)

ref_a = pd.DataFrame({'계수': [-12.027, -0.503, -2.072, 9.834],
                      'chi2': [39.27, 8.06, 11.14, 11.03], 'p': [.000, .005, .001, .001]},
                     index=['Intercept', 'RSIZE', 'EXRET', 'SIGMA'])
ref_b = pd.DataFrame({'계수': [-13.303, -1.982, 3.593, -0.467, -1.809, 5.791],
                      'chi2': [30.79, .88, 6.90, 5.24, 6.52, 2.47],
                      'p': [.000, .348, .009, .022, .011, .116]},
                     index=['Intercept', 'NITA', 'TLTA', 'RSIZE', 'EXRET', 'SIGMA'])
ref = pd.concat({'Panel A 시장': ref_a, 'Panel B 결합': ref_b}, names=['패널', '변수'])
ref_n = pd.DataFrame({'기업': [2894, 2497], 'firm-year': [33621, 28664],
                      '사건': [291, 239], '기업당연수': [11.62, 11.48]},
                     index=pd.Index(['Panel A 시장', 'Panel B 결합'], name='패널'))
display(ref_n)
display(ref)

,기업,firm-year,사건,기업당연수,pseudo-R2
모형,,,,,
M1 시장 [m12],939.0,15970.0,101.0,17.01,0.289
M3 결합 [m12],936.0,15870.0,100.0,16.96,0.350
M1 시장 [d12],927.0,15630.0,96.0,16.86,0.341
M2 회계,939.0,15952.0,101.0,16.99,0.247
M3 결합 [d12],924.0,15539.0,95.0,16.82,0.370
M4 SIGMA제외,936.0,15870.0,100.0,16.96,0.312
M1 시장 [d12_fill],939.0,15970.0,101.0,17.01,0.360
M3 결합 [d12_fill],936.0,15870.0,100.0,16.96,0.390


계수  p_보정없음  p_Shumway  p_클러스터
모형               변수                                          
M1 시장 [m12]      Intercept -14.764   0.000      0.003   0.000
                 RSIZE      -0.836   0.000      0.100   0.000
                 EXRET      -1.801   0.000      0.100   0.000
                 SIGMA       2.587   0.000      0.005   0.000
M3 결합 [m12]      Intercept -13.864   0.000      0.004   0.000
                 NITA       -2.729   0.000      0.289   0.000
                 TLTA        2.940   0.000      0.188   0.000
                 RSIZE      -0.589   0.000      0.224   0.000
                 EXRET      -1.239   0.000      0.250   0.001
                 SIGMA       1.892   0.000      0.080   0.000
M1 시장 [d12]      Intercept -14.524   0.000      0.003   0.000
                 RSIZE      -0.586   0.000      0.249   0.000
                 EXRET      -1.085   0.000      0.257   0.000
                 SIGMA       5.840   0.000      0.003   0.000
M2 회계            Intercept  -7.758   0.000      0.000   0.000
                 NITA       -6.285   0.000      0.006   0.000
                 TLTA        3.806   0.000      0.106   0.000
M3 결합 [d12]      Intercept -14.051   0.000      0.004   0.000
                 NITA       -1.660   0.008      0.515   0.023
                 TLTA        2.016   0.000      0.361   0.002
                 RSIZE      -0.484   0.000      0.327   0.000
                 EXRET      -0.901   0.000      0.351   0.002
                 SIGMA       4.629   0.000      0.041   0.000
M4 SIGMA제외       Intercept -13.590   0.000      0.005   0.000
                 NITA       -4.197   0.000      0.100   0.000
                 TLTA        3.539   0.000      0.124   0.000
                 RSIZE      -0.628   0.000      0.191   0.000
                 EXRET      -1.180   0.000      0.347   0.018
M1 시장 [d12_fill] Intercept -14.325   0.000      0.003   0.000
                 RSIZE      -0.536   0.000      0.273   0.000
                 EXRET      -1.120   0.000      0.230   0.000
                 SIGMA       6.189   0.000      0.002   0.000
M3 결합 [d12_fill] Intercept -13.918   0.000      0.004   0.000
                 NITA       -1.471   0.014      0.549   0.027
                 TLTA        1.923   0.000      0.372   0.002
                 RSIZE      -0.433   0.000      0.363   0.001
                 EXRET      -0.941   0.000      0.316   0.001
                 SIGMA       5.187   0.000      0.021   0.000

모형,M1 시장 [m12],M3 결합 [m12],M1 시장 [d12],M2 회계,M3 결합 [d12],M4 SIGMA제외,M1 시장 [d12_fill],M3 결합 [d12_fill]
변수,,,,,,,,
Intercept,-14.764,-13.864,-14.524,-7.758,-14.051,-13.590,-14.325,-13.918
NITA,NaN,-2.729,NaN,-6.285,-1.660,-4.197,NaN,-1.471
TLTA,NaN,2.940,NaN,3.806,2.016,3.539,NaN,1.923
RSIZE,-0.836,-0.589,-0.586,NaN,-0.484,-0.628,-0.536,-0.433
EXRET,-1.801,-1.239,-1.085,NaN,-0.901,-1.180,-1.120,-0.941
SIGMA,2.587,1.892,5.840,NaN,4.629,NaN,6.189,5.187
사건,101.000,100.000,96.000,101.000,95.000,100.000,101.000,100.000
pseudo-R2,0.289,0.350,0.341,0.247,0.370,0.312,0.360,0.390


모형,M1 시장 [m12],M3 결합 [m12],M1 시장 [d12],M2 회계,M3 결합 [d12],M4 SIGMA제외,M1 시장 [d12_fill],M3 결합 [d12_fill]
변수,,,,,,,,
Intercept,0.0,0.000,0.0,0.0,0.000,0.000,0.0,0.000
NITA,NaN,0.000,NaN,0.0,0.023,0.000,NaN,0.027
TLTA,NaN,0.000,NaN,0.0,0.002,0.000,NaN,0.002
RSIZE,0.0,0.000,0.0,NaN,0.000,0.000,0.0,0.001
EXRET,0.0,0.001,0.0,NaN,0.002,0.018,0.0,0.001
SIGMA,0.0,0.000,0.0,NaN,0.000,NaN,0.0,0.000


계수  p_보정없음  p_Shumway  p_클러스터
SIGMA 정의 변수                                          
m12      Intercept -13.864   0.000      0.004   0.000
         NITA       -2.729   0.000      0.289   0.000
         TLTA        2.940   0.000      0.188   0.000
         RSIZE      -0.589   0.000      0.224   0.000
         EXRET      -1.239   0.000      0.250   0.001
         SIGMA       1.892   0.000      0.080   0.000
d12      Intercept -14.051   0.000      0.004   0.000
         NITA       -1.660   0.008      0.515   0.023
         TLTA        2.016   0.000      0.361   0.002
         RSIZE      -0.484   0.000      0.327   0.000
         EXRET      -0.901   0.000      0.351   0.002
         SIGMA       4.629   0.000      0.041   0.000
d12_fill Intercept -13.918   0.000      0.004   0.000
         NITA       -1.471   0.014      0.549   0.027
         TLTA        1.923   0.000      0.372   0.002
         RSIZE      -0.433   0.000      0.363   0.001
         EXRET      -0.941   0.000      0.316   0.001
         SIGMA       5.187   0.000      0.021   0.000

,기업,firm-year,사건,기업당연수
패널,,,,
Panel A 시장,2894,33621,291,11.62
Panel B 결합,2497,28664,239,11.48


계수   chi2      p
패널         변수                             
Panel A 시장 Intercept -12.027  39.27  0.000
           RSIZE      -0.503   8.06  0.005
           EXRET      -2.072  11.14  0.001
           SIGMA       9.834  11.03  0.001
Panel B 결합 Intercept -13.303  30.79  0.000
           NITA       -1.982   0.88  0.348
           TLTA        3.593   6.90  0.009
           RSIZE      -0.467   5.24  0.022
           EXRET      -1.809   6.52  0.011
           SIGMA       5.791   2.47  0.116

In [11]:
# 9

p = pd.read_parquet('shumway_panel.parquet')
BASE = ['NITA', 'TLTA', 'RSIZE', 'EXRET']
SDEF = {'m12': 'SIGMA_m12_a_f', 'd12': 'SIGMA_d12_a_f', 'd12_fill': 'SIGMA_d12_fill_a_f'}
MODELS = {'M1 시장': ['RSIZE', 'EXRET', 'SIGMA'],
          'M2 회계': ['NITA', 'TLTA'],
          'M3 결합': ['NITA', 'TLTA', 'RSIZE', 'EXRET', 'SIGMA'],
          'M4 SIGMA제외': ['NITA', 'TLTA', 'RSIZE', 'EXRET']}
TRAIN_MIN, TEST_START, TEST_END = 2002, 2008, 2025

def walk(cols, scol):
    use = [(scol if c == 'SIGMA' else c + '_f') for c in cols]
    rows = []
    for t in range(TEST_START, TEST_END + 1):
        tr = p[(p['obs_year'] >= TRAIN_MIN) & (p['obs_year'] < t)].dropna(subset=use + ['y'])
        te = p[p['obs_year'] == t].dropna(subset=use + ['y']).copy()
        if tr['y'].sum() < 5 or len(te) == 0:
            continue
        lo, hi = tr[use].quantile(0.01), tr[use].quantile(0.99)
        Xtr = sm.add_constant(tr[use].clip(lo, hi, axis=1).astype(float))
        Xte = sm.add_constant(te[use].clip(lo, hi, axis=1).astype(float), has_constant='add')
        try:
            r = sm.Logit(tr['y'].astype(float), Xtr).fit(disp=0)
        except Exception:
            continue
        o = te[['code', 'obs_year', 'y']].copy()
        o['ph'] = r.predict(Xte)
        o['dec'] = np.ceil(o.groupby('obs_year')['ph']
                           .rank(pct=True, ascending=False) * 10).clip(1, 10).astype(int)
        rows.append(o)
    return pd.concat(rows, ignore_index=True) if rows else None

res = {}
for sk, scol in SDEF.items():
    for mk, cols in MODELS.items():
        if 'SIGMA' not in cols and sk != 'd12_fill':
            continue
        key = mk if 'SIGMA' not in cols else f'{mk} [{sk}]'
        res[key] = walk(cols, scol)

auc = pd.DataFrame(
    [{'검증 관측': len(r), '검증 사건': int(r['y'].sum()),
      'AUC': round(roc_auc_score(r['y'], r['ph']), 4)} for r in res.values()],
    index=pd.Index(list(res), name='모형'))
base = auc.loc['M4 SIGMA제외', 'AUC']
auc['ΔAUC (vs M4)'] = (auc['AUC'] - base).round(4)
display(auc)

tab = {}
for k, r in res.items():
    ev = r[r['y'] == 1]
    v = ev['dec'].value_counts(normalize=True).mul(100).reindex(range(1, 11)).fillna(0)
    tab[k] = v.round(1)
tab = pd.DataFrame(tab)
tab.loc['6-10'] = tab.loc[6:10].sum().round(1)
tab = tab.drop(index=range(6, 11))
tab.index = pd.Index(['1분위', '2분위', '3분위', '4분위', '5분위', '6-10분위'], name='분위')
display(tab)

ref = pd.DataFrame({'시장전용': [69.0, 10.6, 7.8, 5.0, 2.8, 4.8],
                    '회계+시장': [75.0, 12.5, 6.3, 1.8, 0.9, 3.5]},
                   index=pd.Index(['1분위', '2분위', '3분위', '4분위', '5분위', '6-10분위'],
                                  name='분위'))
display(ref)

m3 = [k for k in res if k.startswith('M3')] + ['M4 SIGMA제외']
rows = []
for k in m3:
    r = res[k]
    g = r.groupby('obs_year').agg(관측=('y', 'size'), 사건=('y', 'sum'))
    rows.append(g['사건'].rename(k))
yr = pd.concat(rows, axis=1).fillna(0).astype(int)
yr['관측'] = res['M3 결합 [d12_fill]'].groupby('obs_year').size()
yr.loc['합계'] = yr.sum()
yr.index.name = '검증연도'
display(yr)

rows = []
for k in m3:
    r = res[k]
    ev = r[r['y'] == 1]
    for lo, hi, lab in [(2008, 2013, '2008~2013'), (2014, 2019, '2014~2019'),
                        (2020, 2025, '2020~2025')]:
        s = r[(r['obs_year'] >= lo) & (r['obs_year'] <= hi)]
        e = s[s['y'] == 1]
        if s['y'].sum() < 2: 
            rows.append({'모형': k, '구간': lab, '관측': len(s), '사건': int(s['y'].sum()),
                         'AUC': np.nan, '1분위 %': np.nan})
            continue
        rows.append({'모형': k, '구간': lab, '관측': len(s), '사건': int(s['y'].sum()),
                     'AUC': round(roc_auc_score(s['y'], s['ph']), 4),
                     '1분위 %': round((e['dec'] == 1).mean() * 100, 1)})
per = pd.DataFrame(rows).set_index(['모형', '구간'])
display(per)

pd.concat([r.assign(model=k) for k, r in res.items()],
          ignore_index=True).to_parquet('shumway_oos.parquet')

,검증 관측,검증 사건,AUC,ΔAUC (vs M4)
모형,,,,
M1 시장 [m12],12340,52,0.9364,-0.0044
M3 결합 [m12],12281,51,0.9604,0.0196
M1 시장 [d12],12065,48,0.9361,-0.0047
M3 결합 [d12],12010,47,0.9519,0.0111
M1 시장 [d12_fill],12340,52,0.9541,0.0133
M2 회계,12349,52,0.8622,-0.0786
M3 결합 [d12_fill],12281,51,0.9727,0.0319
M4 SIGMA제외,12281,51,0.9408,0.0000


,M1 시장 [m12],M3 결합 [m12],M1 시장 [d12],M3 결합 [d12],M1 시장 [d12_fill],M2 회계,M3 결합 [d12_fill],M4 SIGMA제외
분위,,,,,,,,
1분위,78.8,84.3,77.1,85.1,90.4,65.4,90.2,76.5
2분위,9.6,11.8,12.5,8.5,3.8,7.7,5.9,13.7
3분위,7.7,3.9,6.2,4.3,1.9,11.5,3.9,5.9
4분위,0.0,0.0,2.1,0.0,1.9,0.0,0.0,2.0
5분위,1.9,0.0,0.0,0.0,0.0,7.7,0.0,0.0
6-10분위,1.9,0.0,2.1,2.1,1.9,7.6,0.0,2.0


,시장전용,회계+시장
분위,,
1분위,69.0,75.0
2분위,10.6,12.5
3분위,7.8,6.3
4분위,5.0,1.8
5분위,2.8,0.9
6-10분위,4.8,3.5


,M3 결합 [m12],M3 결합 [d12],M3 결합 [d12_fill],M4 SIGMA제외,관측
검증연도,,,,,
2008,1,1,1,1,617
2009,12,12,12,12,632
2010,11,10,11,11,634
2011,4,4,4,4,637
2012,5,4,5,5,652
2013,2,2,2,2,659
2014,0,0,0,0,666
2015,3,3,3,3,667
2016,2,1,2,2,671


관측  사건     AUC  1분위 %
모형               구간                                
M3 결합 [m12]      2008~2013  3831  35  0.9596   82.9
                 2014~2019  4066   9  0.9733  100.0
                 2020~2025  4384   7  0.9142   71.4
M3 결합 [d12]      2008~2013  3728  33  0.9691   90.9
                 2014~2019  3963   7  0.9641   85.7
                 2020~2025  4319   7  0.8448   57.1
M3 결합 [d12_fill] 2008~2013  3831  35  0.9684   88.6
                 2014~2019  4066   9  0.9811  100.0
                 2020~2025  4384   7  0.9512   85.7
M4 SIGMA제외       2008~2013  3831  35  0.9450   77.1
                 2014~2019  4066   9  0.9729  100.0
                 2020~2025  4384   7  0.8424   42.9

In [22]:
# 10

B = 2000
SEED = 42
ORDER = ['M1 시장 [m12]', 'M1 시장 [d12]', 'M1 시장 [d12_fill]', 'M2 회계',
         'M3 결합 [m12]', 'M3 결합 [d12]', 'M3 결합 [d12_fill]', 'M4 SIGMA제외']

oos = pd.read_parquet('shumway_oos.parquet')
ORDER = [m for m in ORDER if m in set(oos['model'])]

W = oos.pivot_table(index=['code', 'obs_year'], columns='model', values='ph')
Y = oos.groupby(['code', 'obs_year'])['y'].first()
comp = W[ORDER].notna().all(axis=1)

idx = W.index[comp]
codes = idx.get_level_values('code').to_numpy()
y = Y.loc[idx].to_numpy().astype(int)
S = W.loc[idx, ORDER].to_numpy(dtype=float)

def auc_fast(yy, ss):
    n1 = int(yy.sum()); n0 = len(yy) - n1
    if n1 == 0 or n0 == 0:
        return np.nan
    r = stats.rankdata(ss)
    return (r[yy == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

uc, inv = np.unique(codes, return_inverse=True)
srt = np.argsort(inv, kind='stable')
st = np.searchsorted(inv[srt], np.arange(len(uc)))
en = np.searchsorted(inv[srt], np.arange(len(uc)), side='right')
firm_idx = [srt[st[i]:en[i]] for i in range(len(uc))]

rng = np.random.default_rng(SEED)
A = np.full((B, len(ORDER)), np.nan)
for b in range(B):
    pick = rng.integers(0, len(uc), len(uc))
    ii = np.concatenate([firm_idx[i] for i in pick])
    yy = y[ii]
    if yy.sum() < 2:
        continue
    for j in range(len(ORDER)):
        A[b, j] = auc_fast(yy, S[ii, j])
ok = ~np.isnan(A).any(axis=1)
A = A[ok]

rows = []
for m in ORDER:
    s = oos[oos['model'] == m]
    rows.append({'모형': m, '원 관측': len(s), '원 사건': int(s['y'].sum())})
t0 = pd.DataFrame(rows).set_index('모형')
t0['공통 관측'] = len(y)
t0['공통 사건'] = int(y.sum())
t0['탈락 관측'] = t0['원 관측'] - len(y)
display(t0)

rows = []
for j, m in enumerate(ORDER):
    a = auc_fast(y, S[:, j])
    bs = A[:, j]
    rows.append({'모형': m, 'AUC': round(a, 4),
                 '부트 평균': round(bs.mean(), 4), '부트 SD': round(bs.std(ddof=1), 4),
                 'CI 하한': round(np.percentile(bs, 2.5), 4),
                 'CI 상한': round(np.percentile(bs, 97.5), 4)})
t1 = pd.DataFrame(rows).set_index('모형')
display(t1)

def delta_table(ref):
    k = ORDER.index(ref)
    rows = []
    for j, m in enumerate(ORDER):
        if m == ref:
            continue
        d0 = auc_fast(y, S[:, j]) - auc_fast(y, S[:, k])
        d = A[:, j] - A[:, k]
        p = 2 * min((d <= 0).mean(), (d >= 0).mean())
        rows.append({'모형': m, 'ΔAUC': round(d0, 4),
                     'CI 하한': round(np.percentile(d, 2.5), 4),
                     'CI 상한': round(np.percentile(d, 97.5), 4),
                     'p': round(min(p, 1.0), 3),
                     '0 배제': '예' if np.percentile(d, 2.5) > 0 or np.percentile(d, 97.5) < 0 else '아니오'})
    return pd.DataFrame(rows).set_index('모형')

t2 = delta_table('M4 SIGMA제외')
display(t2)

t3 = delta_table('M3 결합 [m12]')
display(t3)

ov = pd.DataFrame({'값': [B, len(A), len(uc), len(y), int(y.sum()),
                          round(float(A.shape[0]) / B, 3)]},
                  index=pd.Index(['재추출 횟수', '유효 재추출', '기업', '공통 관측',
                                  '공통 사건', '유효 비율'], name='항목'))
display(ov)

,원 관측,원 사건,공통 관측,공통 사건,탈락 관측
모형,,,,,
M1 시장 [m12],12340,52,12010,47,330
M1 시장 [d12],12065,48,12010,47,55
M1 시장 [d12_fill],12340,52,12010,47,330
M2 회계,12349,52,12010,47,339
M3 결합 [m12],12281,51,12010,47,271
M3 결합 [d12],12010,47,12010,47,0
M3 결합 [d12_fill],12281,51,12010,47,271
M4 SIGMA제외,12281,51,12010,47,271


,AUC,부트 평균,부트 SD,CI 하한,CI 상한
모형,,,,,
M1 시장 [m12],0.9441,0.9440,0.0152,0.9105,0.9705
M1 시장 [d12],0.9406,0.9409,0.0182,0.9017,0.9730
M1 시장 [d12_fill],0.9694,0.9696,0.0097,0.9475,0.9860
M2 회계,0.8768,0.8778,0.0268,0.8220,0.9291
M3 결합 [m12],0.9589,0.9588,0.0098,0.9380,0.9756
M3 결합 [d12],0.9519,0.9523,0.0156,0.9186,0.9783
M3 결합 [d12_fill],0.9735,0.9737,0.0078,0.9556,0.9864
M4 SIGMA제외,0.9376,0.9378,0.0161,0.9022,0.9654


,ΔAUC,CI 하한,CI 상한,p,0 배제
모형,,,,,
M1 시장 [m12],0.0065,-0.0245,0.0406,0.732,아니오
M1 시장 [d12],0.0030,-0.0203,0.0260,0.824,아니오
M1 시장 [d12_fill],0.0318,0.0054,0.0670,0.017,예
M2 회계,-0.0608,-0.1014,-0.0219,0.003,예
M3 결합 [m12],0.0213,0.0039,0.0495,0.003,예
M3 결합 [d12],0.0143,-0.0012,0.0315,0.069,아니오
M3 결합 [d12_fill],0.0359,0.0125,0.0690,0.001,예


,ΔAUC,CI 하한,CI 상한,p,0 배제
모형,,,,,
M1 시장 [m12],-0.0147,-0.0351,0.0008,0.075,아니오
M1 시장 [d12],-0.0183,-0.0537,0.0121,0.285,아니오
M1 시장 [d12_fill],0.0105,-0.0051,0.0285,0.193,아니오
M2 회계,-0.0821,-0.1277,-0.0388,0.000,예
M3 결합 [d12],-0.0069,-0.0372,0.0177,0.707,아니오
M3 결합 [d12_fill],0.0147,0.0037,0.0300,0.003,예
M4 SIGMA제외,-0.0213,-0.0495,-0.0039,0.003,예


,값
항목,
재추출 횟수,2000.0
유효 재추출,2000.0
기업,848.0
공통 관측,12010.0
공통 사건,47.0
유효 비율,1.0


In [24]:
# 11

B = 2000
SEED = 42
BASE = ['NITA', 'TLTA', 'RSIZE', 'EXRET']
SDEF = {'m12': 'SIGMA_m12_a_f', 'd12': 'SIGMA_d12_a_f', 'd12_fill': 'SIGMA_d12_fill_a_f'}
TRAIN_MIN, TEST_START = 2002, 2025

p = pd.read_parquet('shumway_panel.parquet')

def walk2(cols, train_mask=None, test_mask=None):
    d = p.dropna(subset=cols + ['y'])
    tr_pool = d if train_mask is None else d[train_mask.reindex(d.index).fillna(False)]
    te_pool = d if test_mask is None else d[test_mask.reindex(d.index).fillna(False)]
    rows = []
    for t in range(2008, int(p['obs_year'].max()) + 1):
        tr = tr_pool[(tr_pool['obs_year'] >= TRAIN_MIN) & (tr_pool['obs_year'] < t)]
        te = te_pool[te_pool['obs_year'] == t]
        if tr['y'].sum() < 5 or len(te) == 0:
            continue
        lo, hi = tr[cols].quantile(0.01), tr[cols].quantile(0.99)
        Xtr = sm.add_constant(tr[cols].clip(lo, hi, axis=1).astype(float))
        Xte = sm.add_constant(te[cols].clip(lo, hi, axis=1).astype(float), has_constant='add')
        try:
            r = sm.Logit(tr['y'].astype(float), Xtr).fit(disp=0)
        except Exception:
            continue
        o = te[['code', 'obs_year', 'y']].copy()
        o['ph'] = r.predict(Xte)
        rows.append(o)
    return pd.concat(rows, ignore_index=True)

d12_ok = p[[c + '_f' for c in BASE] + [SDEF['d12']]].notna().all(axis=1)

VAR = {
    'A. d12 (원래)':            (BASE, SDEF['d12'],      None),
    'B. d12_fill (원래)':       (BASE, SDEF['d12_fill'], None),
    'C. d12_fill, 훈련=d12표본': (BASE, SDEF['d12_fill'], d12_ok),
    'D. m12 (원래)':            (BASE, SDEF['m12'],      None),
    'E. m12, 훈련=d12표본':      (BASE, SDEF['m12'],      d12_ok),
}
res2 = {}
for k, (b, s, tm) in VAR.items():
    res2[k] = walk2([c + '_f' for c in b] + [s], train_mask=tm)

rows = []
for k, (b, s, tm) in VAR.items():
    d = p.dropna(subset=[c + '_f' for c in b] + [s, 'y'])
    if tm is not None:
        d = d[tm.reindex(d.index).fillna(False)]
    tr = d[(d['obs_year'] >= TRAIN_MIN) & (d['obs_year'] <= 2024)]
    r = res2[k]
    rows.append({'변형': k, '훈련 관측': len(tr), '훈련 사건': int(tr['y'].sum()),
                 '검증 관측': len(r), '검증 사건': int(r['y'].sum()),
                 'AUC': round(roc_auc_score(r['y'], r['ph']), 4)})
t4 = pd.DataFrame(rows).set_index('변형')
display(t4)

def boot_pairs(res_dict, keys):
    W = pd.concat([res_dict[k].assign(model=k) for k in keys], ignore_index=True)
    P = W.pivot_table(index=['code', 'obs_year'], columns='model', values='ph')
    Y = W.groupby(['code', 'obs_year'])['y'].first()
    m = P[keys].notna().all(axis=1)
    idx = P.index[m]
    cd = idx.get_level_values('code').to_numpy()
    yy = Y.loc[idx].to_numpy().astype(int)
    SS = P.loc[idx, keys].to_numpy(dtype=float)
    uc_, inv_ = np.unique(cd, return_inverse=True)
    sr = np.argsort(inv_, kind='stable')
    a0 = np.searchsorted(inv_[sr], np.arange(len(uc_)))
    a1 = np.searchsorted(inv_[sr], np.arange(len(uc_)), side='right')
    fi = [sr[a0[i]:a1[i]] for i in range(len(uc_))]
    rng_ = np.random.default_rng(SEED)
    AA = np.full((B, len(keys)), np.nan)
    for b_ in range(B):
        pk = rng_.integers(0, len(uc_), len(uc_))
        ii = np.concatenate([fi[i] for i in pk])
        v = yy[ii]
        if v.sum() < 2:
            continue
        for j in range(len(keys)):
            AA[b_, j] = auc_fast(v, SS[ii, j])
    AA = AA[~np.isnan(AA).any(axis=1)]
    return yy, SS, AA, len(uc_)

def dtab(yy, SS, AA, keys, ref):
    k = keys.index(ref)
    rows = []
    for j, m in enumerate(keys):
        if m == ref:
            continue
        d0 = auc_fast(yy, SS[:, j]) - auc_fast(yy, SS[:, k])
        dd = AA[:, j] - AA[:, k]
        pv = 2 * min((dd <= 0).mean(), (dd >= 0).mean())
        rows.append({'비교': m, 'ΔAUC': round(d0, 4),
                     'CI 하한': round(np.percentile(dd, 2.5), 4),
                     'CI 상한': round(np.percentile(dd, 97.5), 4),
                     'p': round(min(pv, 1.0), 3),
                     '0 배제': '예' if np.percentile(dd, 2.5) > 0 or np.percentile(dd, 97.5) < 0 else '아니오'})
    return pd.DataFrame(rows).set_index('비교')

K1 = ['A. d12 (원래)', 'B. d12_fill (원래)', 'C. d12_fill, 훈련=d12표본',
      'D. m12 (원래)', 'E. m12, 훈련=d12표본']
y1, S1, A1, n1 = boot_pairs(res2, K1)
rows = []
for j, m in enumerate(K1):
    bs = A1[:, j]
    rows.append({'변형': m, 'AUC(공통)': round(auc_fast(y1, S1[:, j]), 4),
                 'CI 하한': round(np.percentile(bs, 2.5), 4),
                 'CI 상한': round(np.percentile(bs, 97.5), 4)})
t5 = pd.DataFrame(rows).set_index('변형')
t5.insert(0, '공통 관측', len(y1))
t5.insert(1, '공통 사건', int(y1.sum()))
display(t5)

t6 = dtab(y1, S1, A1, K1, 'A. d12 (원래)')
display(t6)

t7 = dtab(y1, S1, A1, K1, 'D. m12 (원래)')
display(t7)

,훈련 관측,훈련 사건,검증 관측,검증 사건,AUC
변형,,,,,
A. d12 (원래),14796,93,12010,47,0.9519
B. d12_fill (원래),15122,98,12281,51,0.9727
"C. d12_fill, 훈련=d12표본",14796,93,12281,51,0.9725
D. m12 (원래),15122,98,12281,51,0.9604
"E. m12, 훈련=d12표본",14796,93,12281,51,0.9604


,공통 관측,공통 사건,AUC(공통),CI 하한,CI 상한
변형,,,,,
A. d12 (원래),12010,47,0.9519,0.9186,0.9783
B. d12_fill (원래),12010,47,0.9735,0.9556,0.9864
"C. d12_fill, 훈련=d12표본",12010,47,0.9732,0.9552,0.9864
D. m12 (원래),12010,47,0.9589,0.9380,0.9756
"E. m12, 훈련=d12표본",12010,47,0.9588,0.9379,0.9756


,ΔAUC,CI 하한,CI 상한,p,0 배제
비교,,,,,
B. d12_fill (원래),0.0216,0.0015,0.0519,0.006,예
"C. d12_fill, 훈련=d12표본",0.0213,0.0013,0.0518,0.007,예
D. m12 (원래),0.0069,-0.0177,0.0372,0.707,아니오
"E. m12, 훈련=d12표본",0.0069,-0.0180,0.0373,0.713,아니오


,ΔAUC,CI 하한,CI 상한,p,0 배제
비교,,,,,
A. d12 (원래),-0.0069,-0.0372,0.0177,0.707,아니오
B. d12_fill (원래),0.0147,0.0037,0.0300,0.003,예
"C. d12_fill, 훈련=d12표본",0.0144,0.0033,0.0299,0.005,예
"E. m12, 훈련=d12표본",-0.0000,-0.0009,0.0008,0.915,아니오


In [25]:
# 12

M1 = {'M1 시장 [m12]', 'M1 시장 [d12]', 'M1 시장 [d12_fill]'}
K2 = [m for m in ORDER if m in M1]

Wm = oos[oos['model'].isin(K2)]
Pm = Wm.pivot_table(index=['code', 'obs_year'], columns='model', values='ph')
Ym = Wm.groupby(['code', 'obs_year'])['y'].first()
mm = Pm[K2].notna().all(axis=1)
idm = Pm.index[mm]
cdm = idm.get_level_values('code').to_numpy()
ym = Ym.loc[idm].to_numpy().astype(int)
Sm = Pm.loc[idm, K2].to_numpy(dtype=float)

ucm, invm = np.unique(cdm, return_inverse=True)
srm = np.argsort(invm, kind='stable')
b0 = np.searchsorted(invm[srm], np.arange(len(ucm)))
b1 = np.searchsorted(invm[srm], np.arange(len(ucm)), side='right')
fim = [srm[b0[i]:b1[i]] for i in range(len(ucm))]

rngm = np.random.default_rng(SEED)
Am = np.full((B, len(K2)), np.nan)
for b_ in range(B):
    pk = rngm.integers(0, len(ucm), len(ucm))
    ii = np.concatenate([fim[i] for i in pk])
    v = ym[ii]
    if v.sum() < 2:
        continue
    for j in range(len(K2)):
        Am[b_, j] = auc_fast(v, Sm[ii, j])
Am = Am[~np.isnan(Am).any(axis=1)]

rows = []
for j, m in enumerate(K2):
    bs = Am[:, j]
    rows.append({'모형': m, 'AUC': round(auc_fast(ym, Sm[:, j]), 4),
                 'CI 하한': round(np.percentile(bs, 2.5), 4),
                 'CI 상한': round(np.percentile(bs, 97.5), 4)})
t8 = pd.DataFrame(rows).set_index('모형')
t8.insert(0, '공통 관측', len(ym))
t8.insert(1, '공통 사건', int(ym.sum()))
display(t8)

t9 = dtab(ym, Sm, Am, K2, 'M1 시장 [m12]')
display(t9)

,공통 관측,공통 사건,AUC,CI 하한,CI 상한
모형,,,,,
M1 시장 [m12],12065,48,0.9356,0.8983,0.9656
M1 시장 [d12],12065,48,0.9361,0.8974,0.9667
M1 시장 [d12_fill],12065,48,0.9549,0.9165,0.9826


,ΔAUC,CI 하한,CI 상한,p,0 배제
비교,,,,,
M1 시장 [d12],0.0005,-0.0413,0.0380,0.969,아니오
M1 시장 [d12_fill],0.0193,-0.0030,0.0456,0.105,아니오


In [37]:
# 13

FVOL_COLS = ['actq', 'rectq', 'invtq', 'ppentq', 'atq', 'lctq', 'dlcq',
             'apq', 'dlttq', 'ltq', 'seqq', 'xoprq', 'cogsq', 'xsgaq']

def compute_fvol(df, deflator='saleq', fvol_cols=FVOL_COLS, diff_lag=4, win=8, min_obs=4):
    d = df.sort_values(['code', 'date']).reset_index(drop=True).copy()
    d['xoprq'] = d['cogsq'] + d['xsgaq']
    pq = pd.PeriodIndex(d['date'], freq='Q')
    d['qidx'] = pq.year * 4 + (pq.quarter - 1)
    g = d.groupby('code', sort=False)
    has_lag = (d['qidx'] - g['qidx'].shift(diff_lag)) == diff_lag
    for c in fvol_cols:
        d[f'd_{c}'] = np.where(has_lag, d[c] - g[c].shift(diff_lag), np.nan)
    g = d.groupby('code', sort=False)
    for c in fvol_cols:
        d[f'std_{c}'] = (g[f'd_{c}'].apply(lambda s: s.rolling(win, min_periods=min_obs).std())
                         .reset_index(level=0, drop=True))
    d['defl_pos'] = d[deflator].where(d[deflator] > 0)
    g = d.groupby('code', sort=False)
    d['avg_defl'] = (g['defl_pos'].apply(lambda s: s.rolling(win, min_periods=min_obs).mean())
                     .reset_index(level=0, drop=True))
    d['avg_defl'] = d['avg_defl'].where(d['avg_defl'] > 0)
    ind = []
    for c in fvol_cols:
        d[f'fvol_{c}'] = d[f'std_{c}'] / d['avg_defl']
        ind.append(f'fvol_{c}')
    return d, ind

def finalize_fvol(d, ind, fvol_cols=FVOL_COLS, min_valid=10, suffix=''):
    d = d.copy()
    rcols = [f'rank_{c}' for c in fvol_cols]
    ranked = d.groupby('date')[ind].rank(pct=True)
    ranked.columns = rcols
    d = pd.concat([d, ranked], axis=1)
    d['n_valid'] = d[rcols].notna().sum(axis=1)
    col = f'FVOL{suffix}'
    d[col] = d[rcols].mean(axis=1)
    d.loc[d['n_valid'] < min_valid, col] = np.nan
    return d, col

fin = pd.read_parquet('financials.parquet')
fin['date'] = pd.to_datetime(fin['date'])
fv, ind = compute_fvol(fin, deflator='saleq')
fv, _ = finalize_fvol(fv, ind, suffix='')

qq = pd.PeriodIndex(fv['date'], freq='Q')
fvol_a = fv.loc[qq.quarter == 2, ['code', 'date', 'FVOL', 'n_valid']].copy()
fvol_a['obs_year'] = fvol_a['date'].dt.year + 1
fvol_a = fvol_a[['code', 'obs_year', 'FVOL', 'n_valid']].sort_values(['code', 'obs_year'])

P = p.merge(fvol_a, on=['code', 'obs_year'], how='left')
lo, hi = P['FVOL'].quantile([.01, .99])
P['FVOL_w'] = P['FVOL'].clip(lo, hi)

BASE = ['NITA', 'TLTA', 'RSIZE', 'EXRET']
BW = [c + '_w' for c in BASE]
BF = [c + '_f' for c in BASE]
SIGW, SIGF = 'SIGMA_d12_fill_a_w', 'SIGMA_d12_fill_a_f'

rows = []
for lab, m in [('5변수 완비', P[BW + [SIGW]].notna().all(axis=1)),
               ('FVOL 유효', P['FVOL'].notna()),
               ('5변수 + FVOL', P[BW + [SIGW, 'FVOL']].notna().all(axis=1))]:
    rows.append({'표본': lab, '행': int(m.sum()), '사건': int(P.loc[m, 'y'].sum()),
                 '사건률 %': round(P.loc[m, 'y'].mean() * 100, 2)})
f1 = pd.DataFrame(rows).set_index('표본')
display(f1)

u = P.dropna(subset=['FVOL', 'y'])
rows = [{'변수': 'FVOL', '부호': '+', '관측': len(u), '사건': int(u['y'].sum()),
         'AUC': round(roc_auc_score(u['y'], u['FVOL']), 4),
         '사건 중앙': round(u.loc[u['y'] == 1, 'FVOL'].median(), 4),
         '정상 중앙': round(u.loc[u['y'] == 0, 'FVOL'].median(), 4)}]
for c, s in [(SIGW, 1), ('NITA_w', -1), ('TLTA_w', 1), ('RSIZE_w', -1), ('EXRET_w', -1)]:
    v = P.dropna(subset=[c, 'y'])
    rows.append({'변수': c, '부호': '+' if s > 0 else '−', '관측': len(v),
                 '사건': int(v['y'].sum()),
                 'AUC': round(roc_auc_score(v['y'], s * v[c]), 4),
                 '사건 중앙': round(v.loc[v['y'] == 1, c].median(), 4),
                 '정상 중앙': round(v.loc[v['y'] == 0, c].median(), 4)})
f2 = pd.DataFrame(rows).set_index('변수')
display(f2)

cm = P[BW + [SIGW, 'FVOL']].notna().all(axis=1)
f3 = P.loc[cm, BW + [SIGW, 'FVOL']].corr().round(3)
f3.index.name = '변수'
display(f3)

,행,사건,사건률 %
표본,,,
5변수 완비,15870,100,0.63
FVOL 유효,15137,83,0.55
5변수 + FVOL,15071,81,0.54


,부호,관측,사건,AUC,사건 중앙,정상 중앙
변수,,,,,,
FVOL,+,15137,83,0.9027,0.8632,0.4790
SIGMA_d12_fill_a_w,+,16261,102,0.9280,0.9354,0.4039
NITA_w,−,15952,101,0.8530,-0.2087,0.0281
TLTA_w,+,16286,102,0.7932,0.8205,0.4715
RSIZE_w,−,16313,102,0.8230,-10.3804,-8.7969
EXRET_w,−,15970,101,0.8057,-0.5106,-0.0394


,NITA_w,TLTA_w,RSIZE_w,EXRET_w,SIGMA_d12_fill_a_w,FVOL
변수,,,,,,
NITA_w,1.000,-0.333,0.296,0.155,-0.397,-0.386
TLTA_w,-0.333,1.000,-0.043,-0.025,0.273,0.164
RSIZE_w,0.296,-0.043,1.000,0.162,-0.261,-0.212
EXRET_w,0.155,-0.025,0.162,1.000,0.196,-0.074
SIGMA_d12_fill_a_w,-0.397,0.273,-0.261,0.196,1.000,0.354
FVOL,-0.386,0.164,-0.212,-0.074,0.354,1.000


In [38]:
# 14

CM = P[BW + [SIGW, 'FVOL']].notna().all(axis=1) & P['y'].notna()
W = P[CM].copy()
MODELS = {'M4 SIGMA제외': BW, 'M3 결합': BW + [SIGW],
          '4변수 + FVOL': BW + ['FVOL_w'], 'M5 결합 + FVOL': BW + [SIGW, 'FVOL_w']}
rows, coefs = [], {}
for k, cols in MODELS.items():
    X = sm.add_constant(W[cols].astype(float))
    yv = W['y'].astype(float)
    m = sm.Logit(yv, X).fit(disp=0)
    g = pd.factorize(W['code'])[0]
    mc = sm.Logit(yv, X).fit(disp=0, cov_type='cluster',
                             cov_kwds={'groups': g, 'use_correction': True})
    coefs[k] = pd.DataFrame({
        '계수': m.params.round(3),
        'p_클러스터': pd.Series(stats.chi2.sf(np.asarray(mc.tvalues) ** 2, 1),
                             index=X.columns).round(3)})
    rows.append({'모형': k, '행': len(W), '사건': int(yv.sum()),
                 'pseudo-R2': round(m.prsquared, 4),
                 '표본내 AUC': round(roc_auc_score(yv, m.predict(X)), 4)})
f4 = pd.DataFrame(rows).set_index('모형')
display(f4)
f5 = pd.concat(coefs, names=['모형', '변수'])
display(f5)

MODELS_F = {'M4 SIGMA제외': BF, 'M3 결합': BF + [SIGF],
            '4변수 + FVOL': BF + ['FVOL'], 'M5 결합 + FVOL': BF + [SIGF, 'FVOL']}
CMF = P[BF + [SIGF, 'FVOL']].notna().all(axis=1) & P['y'].notna()

def walk3(cols):
    d = P[CMF]
    rows = []
    for t in range(2008, int(P['obs_year'].max()) + 1):
        tr = d[(d['obs_year'] >= 2002) & (d['obs_year'] < t)]
        te = d[d['obs_year'] == t]
        if tr['y'].sum() < 5 or len(te) == 0:
            continue
        lo_, hi_ = tr[cols].quantile(0.01), tr[cols].quantile(0.99)
        Xtr = sm.add_constant(tr[cols].clip(lo_, hi_, axis=1).astype(float))
        Xte = sm.add_constant(te[cols].clip(lo_, hi_, axis=1).astype(float), has_constant='add')
        try:
            r = sm.Logit(tr['y'].astype(float), Xtr).fit(disp=0)
        except Exception:
            continue
        o = te[['code', 'obs_year', 'y']].copy()
        o['ph'] = r.predict(Xte)
        o['dec'] = np.ceil(o.groupby('obs_year')['ph']
                           .rank(pct=True, ascending=False) * 10).clip(1, 10).astype(int)
        rows.append(o)
    return pd.concat(rows, ignore_index=True)

resF = {k: walk3(c) for k, c in MODELS_F.items()}
rows = []
for k, r in resF.items():
    ev = r[r['y'] == 1]
    rows.append({'모형': k, '검증 관측': len(r), '검증 사건': int(r['y'].sum()),
                 '표본외 AUC': round(roc_auc_score(r['y'], r['ph']), 4),
                 '1분위 %': round((ev['dec'] == 1).mean() * 100, 1),
                 '6-10분위 %': round((ev['dec'] >= 6).mean() * 100, 1)})
f6 = pd.DataFrame(rows).set_index('모형')
display(f6)

KF = list(MODELS_F)
yf, Sf, Af, _ = boot_pairs(resF, KF)
rows = []
for j, k in enumerate(KF):
    bs = Af[:, j]
    rows.append({'모형': k, 'AUC': round(auc_fast(yf, Sf[:, j]), 4),
                 'CI 하한': round(np.percentile(bs, 2.5), 4),
                 'CI 상한': round(np.percentile(bs, 97.5), 4)})
f7 = pd.DataFrame(rows).set_index('모형')
f7.insert(0, '공통 관측', len(yf))
f7.insert(1, '공통 사건', int(yf.sum()))
display(f7)

f8 = dtab(yf, Sf, Af, KF, 'M3 결합')
display(f8)

f9 = dtab(yf, Sf, Af, KF, 'M4 SIGMA제외')
display(f9)

,행,사건,pseudo-R2,표본내 AUC
모형,,,,
M4 SIGMA제외,15071,81,0.3108,0.9264
M3 결합,15071,81,0.3937,0.9695
4변수 + FVOL,15071,81,0.3835,0.9570
M5 결합 + FVOL,15071,81,0.4303,0.9751


계수  p_클러스터
모형           변수                                
M4 SIGMA제외   const              -14.783   0.000
             NITA_w              -3.811   0.000
             TLTA_w               3.546   0.000
             RSIZE_w             -0.737   0.000
             EXRET_w             -1.318   0.045
M3 결합        const              -15.278   0.000
             NITA_w              -0.869   0.232
             TLTA_w               2.113   0.001
             RSIZE_w             -0.547   0.001
             EXRET_w             -0.972   0.005
             SIGMA_d12_fill_a_w   5.321   0.000
4변수 + FVOL   const              -18.991   0.000
             NITA_w              -0.991   0.221
             TLTA_w               3.666   0.000
             RSIZE_w             -0.673   0.000
             EXRET_w             -1.003   0.055
             FVOL_w               7.215   0.000
M5 결합 + FVOL const              -17.931   0.000
             NITA_w               0.246   0.711
             TLTA_w               2.610   0.000
             RSIZE_w             -0.507   0.003
             EXRET_w             -0.750   0.022
             SIGMA_d12_fill_a_w   4.028   0.000
             FVOL_w               5.357   0.000

,검증 관측,검증 사건,표본외 AUC,1분위 %,6-10분위 %
모형,,,,,
M4 SIGMA제외,12109,51,0.9231,74.5,3.9
M3 결합,12109,51,0.9693,86.3,0.0
4변수 + FVOL,12109,51,0.9641,84.3,0.0
M5 결합 + FVOL,12109,51,0.9802,94.1,0.0


,공통 관측,공통 사건,AUC,CI 하한,CI 상한
모형,,,,,
M4 SIGMA제외,12109,51,0.9231,0.8835,0.9572
M3 결합,12109,51,0.9693,0.9539,0.9822
4변수 + FVOL,12109,51,0.9641,0.9462,0.9786
M5 결합 + FVOL,12109,51,0.9802,0.9721,0.9872


,ΔAUC,CI 하한,CI 상한,p,0 배제
비교,,,,,
M4 SIGMA제외,-0.0462,-0.0815,-0.0183,0.000,예
4변수 + FVOL,-0.0053,-0.0213,0.0106,0.527,아니오
M5 결합 + FVOL,0.0108,0.0031,0.0215,0.000,예


,ΔAUC,CI 하한,CI 상한,p,0 배제
비교,,,,,
M3 결합,0.0462,0.0183,0.0815,0.0,예
4변수 + FVOL,0.0409,0.0199,0.0661,0.0,예
M5 결합 + FVOL,0.0570,0.0274,0.0921,0.0,예


In [39]:
# 15

u = P.dropna(subset=['FVOL', 'y']).copy()
u['dec'] = u.groupby('obs_year')['FVOL'].transform(
    lambda s: pd.qcut(s, 10, labels=False, duplicates='drop') + 1)
base = u['y'].mean() * 100

rows = []
for lab, s in [('전체', u), ('흑자 (NITA>0)', u[u['NITA'] > 0]),
               ('적자 (NITA<=0)', u[u['NITA'] <= 0])]:
    b = s['y'].mean() * 100
    for d_ in range(1, 11):
        t = s[s['dec'] == d_]
        if not len(t):
            continue
        rows.append({'구분': lab, '십분위': d_, '행': len(t), '사건': int(t['y'].sum()),
                     '사건률 %': round(t['y'].mean() * 100, 2),
                     '배수': round(t['y'].mean() * 100 / b, 2) if b > 0 else np.nan})
f10 = pd.DataFrame(rows).set_index(['구분', '십분위'])
display(f10)

rows = []
for lab, s in [('전체', u), ('흑자 (NITA>0)', u[u['NITA'] > 0]),
               ('적자 (NITA<=0)', u[u['NITA'] <= 0])]:
    d10_ = s[s['dec'] == 10]
    rows.append({'구분': lab, '행': len(s), '사건': int(s['y'].sum()),
                 '기저 사건률 %': round(s['y'].mean() * 100, 2),
                 '10분위 사건률 %': round(d10_['y'].mean() * 100, 2),
                 '배수': round(d10_['y'].mean() / s['y'].mean(), 2) if s['y'].mean() > 0 else np.nan,
                 'AUC': round(roc_auc_score(s['y'], s['FVOL']), 4) if s['y'].nunique() > 1 else np.nan})
f11 = pd.DataFrame(rows).set_index('구분')
display(f11)

행  사건  사건률 %     배수
구분           십분위                        
전체           1    1525   0   0.00   0.00
             2    1512   0   0.00   0.00
             3    1511   1   0.07   0.12
             4    1514   1   0.07   0.12
             5    1514   2   0.13   0.24
             6    1506   2   0.13   0.24
             7    1512   0   0.00   0.00
             8    1513   2   0.13   0.24
             9    1510  13   0.86   1.57
             10   1520  62   4.08   7.44
흑자 (NITA>0)  1    1433   0   0.00   0.00
             2    1361   0   0.00   0.00
             3    1311   0   0.00   0.00
             4    1263   0   0.00   0.00
             5    1197   2   0.17   1.34
             6    1123   1   0.09   0.71
             7    1072   0   0.00   0.00
             8     972   0   0.00   0.00
             9     862   1   0.12   0.93
             10    614  10   1.63  13.04
적자 (NITA<=0) 1      92   0   0.00   0.00
             2     151   0   0.00   0.00
             3     200   1   0.50   0.29
             4     251   1   0.40   0.23
             5     316   0   0.00   0.00
             6     383   1   0.26   0.15
             7     438   0   0.00   0.00
             8     539   2   0.37   0.22
             9     645  12   1.86   1.09
             10    901  50   5.55   3.24

,행,사건,기저 사건률 %,10분위 사건률 %,배수,AUC
구분,,,,,,
전체,15137,83,0.55,4.08,7.44,0.9027
흑자 (NITA>0),11208,14,0.12,1.63,13.04,0.8807
적자 (NITA<=0),3916,67,1.71,5.55,3.24,0.8320
